<a href="https://colab.research.google.com/github/ahlamAC235/coursera/blob/main/Eckweisbach_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌲 France Valley — Germany Forest Analysis v3
**Full AOI** (all polygons) | Before: May–Jul 2025 | After: May–Jul 2026
---

In [18]:
!pip install earthengine-api geemap shapely -q

In [19]:
import ee, geemap
ee.Authenticate()
ee.Initialize(project='able-device-485115-u4')


In [20]:
BEFORE_START = '2025-05-01'
BEFORE_END   = '2025-07-31'
AFTER_START  = '2026-05-01'
AFTER_END    = '2026-07-31'
CLOUD_MAX    = 25


In [21]:
from shapely import wkt as shapely_wkt
import json

# Full AOI — paste the WKT string here
WKT = """MULTIPOLYGON (((9.972117125494785 50.548962928810184, 9.972119226994879 50.54894842776895, 9.971552535065838 50.548406057275756, 9.971348498162518 50.54845719740233, 9.971476396616096 50.54872990626758, 9.971514744689236 50.5489114139026, 9.971514631921108 50.54909448858877, 9.971478998221665 50.549285738629, 9.972117125494785 50.548962928810184)), ((9.965230450255005 50.55291783891566, 9.965220074765105 50.55290454888741, 9.9641876251432 50.55322989836861, 9.963938162011113 50.55352469713149, 9.962136719467708 50.554454325322396, 9.960789314098507 50.55468559084697, 9.960782235969127 50.55469711247452, 9.961009549337733 50.55512530995701, 9.96122109592413 50.55539823601111, 9.961352868730744 50.55553190832452, 9.96160886415927 50.555354167703065, 9.962256698955821 50.555001000162754, 9.963079532498863 50.55448790080562, 9.963475726958334 50.55427216693126, 9.96410619262209 50.553986148990205, 9.96476992612526 50.55374881493739, 9.964918422101896 50.55342653705727, 9.965230450255005 50.55291783891566)), ((9.969204834404664 50.55253058594636, 9.969336684814289 50.55242315207362, 9.96950012802093 50.55233456553507, 9.96968846883345 50.55226860138016, 9.970034489607011 50.55218212163534, 9.970362574177496 50.55204226665963, 9.970547986667142 50.55192151835305, 9.970751182470472 50.551725626226386, 9.970828572041937 50.55160289203616, 9.970850856278984 50.551527577475056, 9.970415845749665 50.55143203367608, 9.970335813246002 50.551388988765595, 9.970299596720723 50.55132909241802, 9.970303117162834 50.55128706050182, 9.970327762937572 50.5512484448497, 9.970559910247676 50.551154445610464, 9.970770851290633 50.551033765395296, 9.970691240105873 50.5506832032212, 9.97055590235339 50.55061752960254, 9.97040133815154 50.5505748816159, 9.970234819307432 50.550556907721386, 9.970067104836598 50.550564722311925, 9.969984473086459 50.550578502219196, 9.969038617427085 50.550940069976974, 9.96891311685424 50.551517569515745, 9.968849757534858 50.55169938913394, 9.968643984240723 50.55171917797893, 9.968380273974097 50.55185391562714, 9.968121390043557 50.551860599912786, 9.96775535272728 50.55204143299872, 9.967271528079161 50.55218605189233, 9.9671537338465 50.55215154838415, 9.966538819128287 50.55253050909558, 9.966494858997443 50.55262130459648, 9.966485039843292 50.55271518259273, 9.966511382151333 50.55281882229427, 9.966554605696027 50.55289018479291, 9.96679420879402 50.55289709549555, 9.967111867656392 50.55293987565359, 9.967749563114783 50.55306385941, 9.9675910320979 50.553297426672934, 9.967555852337949 50.55338548749337, 9.968035351046295 50.55348016954306, 9.968343442881823 50.55343170733077, 9.968947879048727 50.553429969178886, 9.968934119785903 50.55312648318595, 9.968972787237488 50.55293265891385, 9.969053701519933 50.552743994733824, 9.969204834404664 50.55253058594636)), ((9.97293609768313 50.552876419376794, 9.972993160624958 50.552816260676124, 9.97303200233501 50.552832770502235, 9.97393495227773 50.55270303718086, 9.973952548509313 50.55229890386451, 9.973924239678615 50.55216172941786, 9.9738126161066 50.55196828173857, 9.973626015598569 50.55180100423787, 9.97346481653743 50.55176579893684, 9.973298199136822 50.551753307050284, 9.973131691562694 50.55176278182233, 9.972545498417677 50.55185577102839, 9.971763247573838 50.55182820960911, 9.97153046767224 50.55180041259515, 9.971307806713622 50.55174914185086, 9.971101103491591 50.551675764726895, 9.97091058361175 50.551581002723445, 9.970812541047879 50.551738933161786, 9.97073179464699 50.55182686116922, 9.971513300749267 50.55206350211172, 9.97187210823989 50.5522099172036, 9.972239771974367 50.552396184491826, 9.971900867723715 50.55247938992564, 9.971680252790701 50.55256842720572, 9.971399570148666 50.55274090685833, 9.971291945532768 50.55283743982052, 9.971293781536707 50.552851503192876, 9.972212630026934 50.55298797313329, 9.97282089661897 50.553099353035456, 9.97293609768313 50.552876419376794)), ((9.978684683521795 50.55421074763552, 9.979016611417581 50.55400737537693, 9.97892817375536 50.553772945580825, 9.978753112099017 50.55347262489744, 9.978580189119132 50.55325870115398, 9.978298412048321 50.55299360692715, 9.977945533685627 50.552759385993255, 9.977716842826325 50.55253873368188, 9.977533495136068 50.55249795971127, 9.975893844447008 50.551972638864214, 9.97577920168582 50.55228866733645, 9.975746140103926 50.55245008448141, 9.975949249566586 50.552495759845165, 9.97605572814601 50.55254970649323, 9.976157327533192 50.55266003591002, 9.976174610550018 50.55278547520059, 9.976098472192396 50.55293329266251, 9.975997054945088 50.553057065178294, 9.975865277616005 50.55316901056136, 9.975597612871024 50.55333490791155, 9.975513005887533 50.55344713817458, 9.978684683521795 50.55421074763552)), ((9.96921366092903 50.55260349825642, 9.969423251770522 50.55299844620522, 9.969517079353306 50.553192116913074, 9.969556039178261 50.55333011432012, 9.96963322781235 50.55346215827458, 9.969813514886305 50.55363757359424, 9.97068677312491 50.554240584983525, 9.970796551439152 50.554393406508396, 9.970812951103575 50.55444835813486, 9.970816540661454 50.5545709657032, 9.970962254165842 50.55458469685094, 9.971166655738552 50.55455074145441, 9.971282184553264 50.554495865812015, 9.971409059550838 50.5543753710117, 9.97147135363446 50.55422879222931, 9.971495494030494 50.553856625129804, 9.971574010518324 50.55351816525096, 9.971579029402296 50.55343761765185, 9.971547076875858 50.55335441052757, 9.971555095362213 50.55327326932795, 9.97170663642813 50.55307389379831, 9.971745784692606 50.55294396118586, 9.971157990988422 50.552862343731235, 9.970045485225347 50.55222080821515, 9.969716348288781 50.552301031200834, 9.969540087625186 50.55236289117605, 9.969321315894529 50.55249444934069, 9.96921366092903 50.55260349825642)), ((9.967537614832832 50.553411048832416, 9.96715484798154 50.55329746581674, 9.966922142126517 50.553196215246984, 9.966652116309465 50.55304036071066, 9.96655208326774 50.55295821487688, 9.966484883298255 50.55287776938993, 9.966311781379726 50.55290430001475, 9.96613522159021 50.55295629348896, 9.965980145561113 50.55296789028027, 9.965300891031388 50.55287798417138, 9.964995125295404 50.553374887837016, 9.9656448075395 50.55337320944387, 9.96683645172857 50.55341130114104, 9.967431409534584 50.553467521936156, 9.967711578453326 50.553446527320574, 9.967537614832832 50.553411048832416)), ((9.978785987932211 50.55271778104914, 9.978365315280362 50.552624769392594, 9.978329780274963 50.552674027692674, 9.977823584903792 50.55257143943067, 9.978002282498236 50.55274177032028, 9.978265969837832 50.552907766705665, 9.978408795786208 50.55302101782308, 9.978616415605917 50.553223168432645, 9.978790742134679 50.553437568389946, 9.978968339883773 50.55373888010053, 9.979076381750236 50.55402082282271, 9.978755635764637 50.55422377367721, 9.979032268544529 50.554294945195345, 9.979280406946295 50.554304582071545, 9.979559429640245 50.55393927483802, 9.97980173184963 50.55367890447979, 9.98007191594372 50.553429687137175, 9.980367610011523 50.553193853962966, 9.981068548774697 50.55293950239913, 9.98086480729776 50.55285464783568, 9.978785987932211 50.55271778104914)), ((9.978732289552783 50.5542798803178, 9.978734863216621 50.55426214544079, 9.978254121154913 50.55413851681662, 9.974537518513667 50.55325525824195, 9.974421061590693 50.55331412383678, 9.974542994790804 50.553430820066716, 9.97462649638417 50.55355448380068, 9.974672024793843 50.553685488619955, 9.974666586314816 50.55388546463764, 9.974498484393079 50.554171394944056, 9.97424174314489 50.554438341176, 9.974251895470093 50.55444780460476, 9.974664164696009 50.55439382062901, 9.975312446256988 50.55426775470766, 9.977142087421948 50.554201770655894, 9.97776579389056 50.55415836243509, 9.978559675066181 50.55427117403619, 9.978732289552783 50.5542798803178)), ((9.969180647409262 50.55433720141425, 9.970294115054921 50.55485030085103, 9.970305907317806 50.554778300314766, 9.970296259127371 50.55463747196674, 9.970244220263664 50.55450083017526, 9.970152730306296 50.554374351066194, 9.969948784407023 50.55421166083732, 9.969846225586695 50.554095908882914, 9.96947379164338 50.553757569669344, 9.969039678114706 50.55343573059761, 9.969052124907204 50.553511759044966, 9.969090928953147 50.553604049168555, 9.969205805946084 50.55417633543167, 9.969180647409262 50.55433720141425)), ((9.969081115692147 50.55426697825432, 9.966883597798702 50.55385924484347, 9.965334461063163 50.55381230243269, 9.965329650405346 50.553827576977966, 9.965521829452811 50.55401459523021, 9.964749973938652 50.554207448592855, 9.964312441624088 50.55428702348271, 9.963859473797553 50.55431491518554, 9.963487118975879 50.55429915722291, 9.96308053093004 50.554521866104146, 9.962289724094848 50.55501583126748, 9.96163607878022 50.555372357740495, 9.96114171776326 50.555699574216796, 9.960758610528258 50.55589818188271, 9.960175425159628 50.55614285751388, 9.960981864164939 50.556583148495996, 9.96143443473671 50.55654424856723, 9.961586954922854 50.55650265548478, 9.961724392569158 50.556441452536745, 9.961838753336721 50.556364040802904, 9.962388565398218 50.55590350338728, 9.962764464919317 50.5556944813987, 9.963257152673314 50.5554706913535, 9.96336514142164 50.555406679753844, 9.963729154719266 50.55511957884502, 9.964187652893362 50.55482162169675, 9.964315360362908 50.55475014351376, 9.964464737488708 50.554697913525786, 9.964740838523994 50.554646951288014, 9.964970199228912 50.5546264792506, 9.965529550843074 50.554638719918096, 9.966316158814557 50.554582092509044, 9.966910454629513 50.55457128964794, 9.968143326099044 50.554402499344995, 9.969117388793425 50.55432055450536, 9.969122180279399 50.554300902324215, 9.969081115692147 50.55426697825432)), ((9.98124470479621 50.55518774284536, 9.98121200807515 50.55524614725904, 9.981191650326492 50.555386746762615, 9.981215823351896 50.55552565550556, 9.981256638606109 50.55560387823289, 9.981581209709109 50.55533300872987, 9.98124470479621 50.55518774284536)), ((9.976073126844716 50.55425819938126, 9.97529461991533 50.5542881509839, 9.97492560677026 50.55436305419119, 9.974927731586462 50.55437753899708, 9.975169208117693 50.55446638202403, 9.975361494168927 50.554592475149285, 9.97541220712112 50.554641077634386, 9.975430385439768 50.55468137579154, 9.975404658648518 50.554734486340124, 9.975350942123523 50.55476343456016, 9.97487676181771 50.554806785599354, 9.974762663188796 50.55479541652048, 9.974654842242623 50.55476459223833, 9.974200101091549 50.554754097877755, 9.974055310072966 50.554772952802395, 9.973921506821444 50.55481474930058, 9.97376307024951 50.554913157973935, 9.97369356418437 50.55499769934812, 9.973628943751475 50.5552459222824, 9.973485847056194 50.55548628535057, 9.973673047514945 50.555662107660524, 9.973904815160132 50.55591395647835, 9.97410617506944 50.55617600358374, 9.9742558853265 50.55641052543279, 9.974636878047354 50.556404460810754, 9.974798294795935 50.55635486987331, 9.974854451203143 50.55630815940844, 9.974885956355964 50.556251583780146, 9.974905439821738 50.5560074970274, 9.974882647259813 50.55579402714956, 9.974965801648658 50.555665602928116, 9.975065066318457 50.555594699998714, 9.975181784995648 50.55554634100539, 9.975520265933564 50.55546863882461, 9.975714394947957 50.55544606958462, 9.976009352753016 50.5554571916224, 9.976286777563985 50.55552101621834, 9.977605903784923 50.556220476734325, 9.978252591444743 50.55608929366442, 9.978696187135883 50.556290334782126, 9.979101225956926 50.55602872539342, 9.979095825512621 50.55601481835705, 9.978906483647805 50.55600143138077, 9.978725400063901 50.55596890872865, 9.978475281766086 50.5558852039558, 9.979217139051077 50.55530708615173, 9.978988158691864 50.55511814279848, 9.978730508263835 50.55495655782124, 9.978294045873081 50.554752807305, 9.97741041079033 50.554293709382904, 9.977260689006185 50.55428120893051, 9.977156566282915 50.55431254872598, 9.976804586782384 50.55460731347909, 9.976705180027453 50.55464033256588, 9.976629731108746 50.55464363578749, 9.97649455331892 50.55460460616338, 9.976274989218943 50.55440577654317, 9.976073126844716 50.55425819938126)), ((9.96839534433658 50.55493676331999, 9.968927403740985 50.55458288702194, 9.969032762076715 50.55447901428071, 9.969097749804318 50.55436336909035, 9.968156051807245 50.5544377451801, 9.966920930437883 50.55460676894823, 9.966318405141644 50.55461804884122, 9.965529531829867 50.55467469673187, 9.964972209807698 50.55466242537244, 9.964569809308065 50.55471323702976, 9.964423623883626 50.55475137785678, 9.964292825597676 50.55481001055749, 9.963781668117237 50.5551410771731, 9.963417991357527 50.555428035921146, 9.963230974562164 50.55554023876019, 9.963745906863462 50.55609667785507, 9.964088491866605 50.555969644795454, 9.966677351230802 50.55525261926497, 9.967874774070768 50.555276454763685, 9.968115563837326 50.555133069575156, 9.96839534433658 50.55493676331999)), ((9.970763708900718 50.5551291959002, 9.970763395677121 50.555113007530146, 9.969170863619782 50.554377972612855, 9.96909408758379 50.55449236728936, 9.968981818411255 50.554602916509786, 9.968651061899804 50.55482455837459, 9.970252185562273 50.55538818916773, 9.970763708900718 50.5551291959002)), ((9.969453966428686 50.55572672700142, 9.96945612782991 50.555744292514525, 9.969555587503345 50.55573073978053, 9.969161206628657 50.55579698694964, 9.966784199164215 50.556782790959936, 9.96731074603507 50.55679719132909, 9.967391331171982 50.556750882335436, 9.968083377196614 50.55672195565474, 9.968844925113368 50.55664034960327, 9.96969033262569 50.55662865186541, 9.97106837123182 50.55664405577148, 9.971100392053662 50.55651549512735, 9.971199755833489 50.556363208586134, 9.971290256714305 50.55631451580329, 9.971367332303553 50.556299671527135, 9.9715232323719 50.55631945560483, 9.972000829172579 50.55657647112622, 9.972795574204175 50.556503418970806, 9.973346516082422 50.556430283966144, 9.973744168741879 50.55640178615763, 9.973445346819863 50.55624650540513, 9.973212255973385 50.55609689405956, 9.973008778823388 50.5559308310726, 9.972759061368563 50.5556524579704, 9.972191140392903 50.55590289815981, 9.971794700105857 50.55603937622513, 9.971775516993034 50.5560140694887, 9.972352394185524 50.55580431695252, 9.972717649144734 50.55563439263187, 9.973219292420533 50.555339407127725, 9.97298708185816 50.555160274654526, 9.972588671356304 50.55488991541342, 9.972353406027828 50.55494510364305, 9.970830726275555 50.555560726975, 9.970307065936677 50.55569147958077, 9.970021644102783 50.555688276828946, 9.969627058374352 50.555720594811405, 9.970030151231086 50.555678305230465, 9.970294495108114 50.555675339639514, 9.970555555592902 50.555621237621935, 9.971003054685637 50.55548151663318, 9.971530038056967 50.55524810665109, 9.972338635908033 50.554930240205564, 9.972551927912903 50.55487515870746, 9.972003121395153 50.55452313807173, 9.970271639243643 50.55542894007334, 9.969837436354187 50.555599025683826, 9.969453966428686 50.55572672700142)), ((9.967798889089682 50.55531899605025, 9.967794979255467 50.55530198086865, 9.966853602685896 50.55529195766056, 9.967360752029377 50.55552166471436, 9.967798889089682 50.55531899605025)), ((9.980844731986009 50.554992993045495, 9.980208168245932 50.5553725090111, 9.980321052786799 50.55541528680163, 9.980496413433254 50.5557110043858, 9.980695671481602 50.55593636779732, 9.981214003803839 50.55564063754354, 9.981142125190273 50.55543306901804, 9.98114588643077 50.55530113606573, 9.981187378455493 50.55516308817196, 9.980844731986009 50.554992993045495)), ((9.968622167989125 50.55600109413912, 9.968620115157812 50.555747970281594, 9.968654812287863 50.55556443819016, 9.96870992413103 50.55546563745711, 9.968722622324727 50.5554120333555, 9.968715033658166 50.555303657630894, 9.968694140998688 50.55524949668145, 9.968622877656612 50.55515232833154, 9.968515629906227 50.55507056990085, 9.968380427354576 50.555009573199385, 9.96786242390096 50.55533543106116, 9.96743328211221 50.55554010363786, 9.967754636307264 50.55566234314406, 9.967876416004831 50.55565754867931, 9.968011502815422 50.55570492459928, 9.968053115109582 50.55574569127597, 9.9680714603101 50.555791552563065, 9.968037280996116 50.55588884288725, 9.96804182312622 50.555930868447284, 9.968065883457692 50.555966283897234, 9.96812884231808 50.5560038024902, 9.96840034570026 50.55609072142659, 9.968622167989125 50.55600109413912)), ((9.976648712224408 50.55575309882409, 9.976249131078736 50.55554059517361, 9.976065186978346 50.55549388655935, 9.975858979492465 50.555471762116916, 9.975548323633545 50.55549206842248, 9.975223030430447 50.55556446250105, 9.975057464241848 50.55564074411863, 9.974960689311118 50.55574450403071, 9.975554256581475 50.5563321205386, 9.976002881746803 50.55683750366718, 9.976152904511805 50.55705149979225, 9.976254614057178 50.557248280061074, 9.976545478838649 50.55724479859556, 9.976797374204583 50.55730594982402, 9.977428540529763 50.557086660297784, 9.978179326305648 50.55705933447762, 9.978844246014873 50.55638619187974, 9.978238303546656 50.55612192716076, 9.97759356622503 50.55625319923279, 9.976648712224408 50.55575309882409)), ((9.960284855778776 50.55603029555985, 9.959772678639823 50.556191257974106, 9.95927730506816 50.55638774961384, 9.958586589532535 50.557002020919874, 9.958273607899995 50.55731568929066, 9.958020975817433 50.55751479149732, 9.958271399202813 50.557583518498205, 9.958534845478571 50.55723662384914, 9.958732930752255 50.55700479105885, 9.958959815629045 50.55678383614552, 9.959213720917743 50.556575456248765, 9.95973724662346 50.55628856207286, 9.960295932406712 50.556057771411595, 9.960284855778776 50.55603029555985)), ((9.958340191979335 50.557590910656515, 9.958343861670803 50.55760625276492, 9.960163022958294 50.55810437461025, 9.96050231853823 50.557791961017365, 9.960783664787835 50.55744537485669, 9.960825667709607 50.55742843770717, 9.960748421365338 50.55737428840685, 9.960582399606599 50.557296711943735, 9.960294228610792 50.557224609067354, 9.958866218674222 50.55718049871053, 9.958565698278846 50.55739508598265, 9.958340191979335 50.557590910656515)), ((9.964078031559982 50.55728257684277, 9.965296658135616 50.557584958318316, 9.965065411959174 50.55728964153163, 9.965037076601234 50.557175289932495, 9.96500621315148 50.55711891015621, 9.964914118239536 50.557015806810675, 9.964786565765827 50.55693028596787, 9.964632221014837 50.55686740101433, 9.96434524380244 50.556661134971534, 9.964082164338015 50.556441706339605, 9.96378365081079 50.556118103758294, 9.962863288875958 50.55645879399182, 9.962691362658221 50.556951284847365, 9.962731224109639 50.55694210302905, 9.964078031559982 50.55728257684277)), ((9.971263036346286 50.557435705522145, 9.971648417067065 50.55748512888713, 9.971848623844075 50.5574898703007, 9.972046808842775 50.55746874288589, 9.972234309976947 50.557422544546895, 9.97247901746089 50.557310522382636, 9.972708502137726 50.55712835436476, 9.973082831434262 50.556895351179854, 9.972822922955395 50.55683224577551, 9.972463570818137 50.55680966941143, 9.972278102184566 50.556852687339436, 9.972135434910426 50.55693830455963, 9.972022169717418 50.55696629230066, 9.971661511014752 50.557016557746174, 9.97141509865755 50.55702002833474, 9.970871165498798 50.556986535425494, 9.970433919950107 50.5569236303874, 9.97015691596842 50.55685364114185, 9.969896050041326 50.55672244869244, 9.969697636374349 50.556655703439745, 9.968853263894244 50.55666712269538, 9.968223943406509 50.55674441139246, 9.968426107452931 50.556822876055634, 9.968745526617674 50.55688768127691, 9.969080189839143 50.556903736305244, 9.969280297802857 50.55688601633991, 9.969409107358134 50.55690917863241, 9.969518683440572 50.55695747317973, 9.969596985194386 50.557025104654365, 9.969674796823861 50.55722218257817, 9.96982891287125 50.55722020044998, 9.970094907894197 50.55724622886312, 9.970248090770362 50.55728717561875, 9.970445602109692 50.55738487264084, 9.970758078453944 50.557391758470565, 9.971263036346286 50.557435705522145)), ((9.966880378475052 50.558780911615834, 9.966591224155824 50.55890210195343, 9.966524460384766 50.55896301468168, 9.966494157581623 50.55903566074403, 9.966497362942958 50.55919561094585, 9.966470200251798 50.5593311309137, 9.96657742047533 50.559301394944065, 9.966830006753556 50.55915526226116, 9.966976459666611 50.55903924060523, 9.967094830487643 50.55891011020887, 9.966880378475052 50.558780911615834)), ((9.984255122992115 50.55786913914598, 9.984230465234598 50.55756969251343, 9.984187552270042 50.557472072124085, 9.983666880372171 50.55669531745743, 9.98352451967273 50.556527504487896, 9.983196282488933 50.55657300606316, 9.982890610011797 50.55665434227577, 9.983935813600091 50.557539921394905, 9.983689549608714 50.55761215230508, 9.983720897417363 50.5577652142412, 9.983688800559817 50.557986652000224, 9.984198230195005 50.55806180467111, 9.984255122992115 50.55786913914598)), ((9.980464057695492 50.55671662098859, 9.98110944483908 50.55690327665649, 9.981620795675742 50.55707891615325, 9.982075108076026 50.55726471938844, 9.982816224567243 50.55763327720774, 9.982828777680066 50.55762809669552, 9.982421245615136 50.55708227386499, 9.98230979402181 50.55698747421151, 9.9817579758257 50.556649019809505, 9.981072298206882 50.55628145030622, 9.980845341631397 50.556124696863435, 9.980643556377053 50.55595164563035, 9.979662704364953 50.55651951331747, 9.980037810406927 50.55661115166947, 9.98016991773664 50.55655242126809, 9.980464057695492 50.55671662098859)), ((9.973591071625503 50.557202672494775, 9.973672482869931 50.5571902403285, 9.97373481565228 50.55715334153025, 9.973758688124871 50.5571158889688, 9.973766004052312 50.55704222871406, 9.973747954376982 50.55693289980903, 9.973697792721081 50.556829233625244, 9.973659389122895 50.556778289161336, 9.97356396275244 50.556732306265246, 9.973284303139035 50.5568428859448, 9.973002441622425 50.55698454489062, 9.972747355889682 50.55714601629735, 9.972523378555941 50.557324523012305, 9.972263885205624 50.557444480861896, 9.97206365345124 50.557494150398405, 9.971851608576658 50.557516788757745, 9.971636803018015 50.55751141894107, 9.971210098189754 50.557457779132655, 9.970125518555012 50.55829883973185, 9.970712582226595 50.55835786135579, 9.970984335611748 50.55836786909804, 9.971250028573527 50.55840270980567, 9.971626656657488 50.558499830745866, 9.971996026170402 50.558662924527155, 9.97259594842161 50.55825394792792, 9.97271398103153 50.55819067420592, 9.972926126759232 50.55813168487538, 9.973163894070396 50.55811977986903, 9.973399097560028 50.55766089667118, 9.973591071625503 50.557202672494775)), ((9.967233103030571 50.55791369230023, 9.967513018118407 50.55776547258763, 9.967535795835738 50.55768765562393, 9.967543829867267 50.55753788730419, 9.967506269031066 50.55734662804071, 9.967459651291627 50.55728823503763, 9.967374834698106 50.557251988313716, 9.967274370351543 50.55724715864869, 9.967182887713445 50.55727449785257, 9.967122876862783 50.55732812160851, 9.967110458312503 50.55739029739961, 9.966439352388226 50.5577384334677, 9.967233103030571 50.55791369230023)), ((9.966080445079642 50.55771186548455, 9.96542917801934 50.55762331375627, 9.965887182012162 50.55802017715546, 9.966388578152289 50.55777536375444, 9.966080445079642 50.55771186548455)), ((9.983656202574887 50.55814769411637, 9.983659116139632 50.55813118971646, 9.983412621814818 50.55798968534006, 9.982875429532305 50.55814886873552, 9.982691517183552 50.55825938985263, 9.98261711964221 50.55835614683224, 9.983656202574887 50.55814769411637)), ((9.98414255667404 50.55838994098834, 9.984157538772019 50.558228487851366, 9.984189643071192 50.55809066680779, 9.983667670196018 50.55801760079955, 9.983643939233017 50.558081703456004, 9.98414255667404 50.55838994098834)), ((9.968828947094114 50.55825212790686, 9.968820959774234 50.55824051401447, 9.96808929907398 50.558148699573906, 9.967477751161232 50.55801389081169, 9.967466538927033 50.55802192121935, 9.967524022365525 50.55814621355939, 9.96754500360926 50.558339593499, 9.967741651146868 50.558449170065884, 9.968113170935313 50.5585515110898, 9.968257086560174 50.55856748815995, 9.968401892115564 50.55855960656764, 9.968602587337122 50.55850440099391, 9.968762500033966 50.55836867222465, 9.968828947094114 50.55825212790686)), ((9.982414279194199 50.558988015226895, 9.982426968807308 50.558995563603986, 9.982885753883322 50.558738606526525, 9.983202319875373 50.55860669451095, 9.983539161804782 50.5584969323591, 9.984040750166853 50.55838754414495, 9.984046900579527 50.558375151921645, 9.983712398424077 50.5581653326328, 9.982380996793376 50.55844631736255, 9.982414279194199 50.558988015226895)), ((9.971947581133563 50.55869400249635, 9.971947312630657 50.55867866178602, 9.971819130487473 50.55861140404635, 9.971596352462914 50.558521476604696, 9.971229996264674 50.55842754099756, 9.970971804321081 50.55839411221864, 9.970709127871912 50.558384923708296, 9.970081723301153 50.558328250592865, 9.96972729072312 50.55860129945987, 9.97060484531839 50.559133638865056, 9.971045046607504 50.55878853801069, 9.971280803791547 50.55884354612474, 9.9715123314326 50.558963933546394, 9.971947581133563 50.55869400249635)), ((9.972876702256606 50.56172739270282, 9.972877872796502 50.561743077634056, 9.973205826211684 50.56189417107462, 9.973437962588845 50.56177412374774, 9.973154468302784 50.56154092676182, 9.972876702256606 50.56172739270282)), ((9.96545698145364 50.561548916875125, 9.965452557958182 50.56156520725663, 9.966173287442377 50.56208942291418, 9.966312552504004 50.5619243964835, 9.966502417889552 50.56175298340669, 9.966282602659007 50.561644365199655, 9.96598835000413 50.56156343559828, 9.96577694884573 50.56153900090498, 9.96545698145364 50.561548916875125)), ((9.978325126888125 50.562058935037285, 9.97802143248536 50.5623296802557, 9.979076103005404 50.562602488093724, 9.979956647434863 50.56266886702399, 9.97999785318102 50.56268888475448, 9.98008087182959 50.56278341432226, 9.980189195312667 50.562861479914865, 9.980317786885948 50.56292583848952, 9.980466555338014 50.56297520723657, 9.980564400882757 50.56287130662271, 9.980661581334166 50.562729162463, 9.980699236906855 50.56264603098566, 9.980712123952877 50.56255229971483, 9.980699326664471 50.56247492662316, 9.980656862327796 50.56238711601461, 9.980429738462906 50.56212102870152, 9.980418109663937 50.56206015943723, 9.980436575328453 50.562004863119725, 9.980481463252413 50.56195405327922, 9.980565457915722 50.56190273618964, 9.980788463232859 50.5618359868481, 9.981012237171276 50.56183489205979, 9.981244483092409 50.56189153635034, 9.981244286112256 50.561764042165706, 9.981271394155977 50.56156794863576, 9.978325126888125 50.562058935037285)), ((9.97430211680075 50.56201996186402, 9.974304493435339 50.562005569330225, 9.974192177504944 50.56189770885684, 9.973858386788802 50.56202193264575, 9.973639436324545 50.56202966843533, 9.973167612806954 50.56215003720337, 9.972952223875339 50.56225059113623, 9.972953431587056 50.5622657972936, 9.97323074259482 50.56241030340639, 9.973144111471106 50.562483976080834, 9.972573813403779 50.56253192586637, 9.972273374080894 50.56232586345827, 9.971524414597548 50.562335749729584, 9.97130451170705 50.56244823310029, 9.971205279019188 50.56254582863933, 9.970808798039245 50.562992489780505, 9.970578002190427 50.563311102617995, 9.970748410764362 50.56336375919978, 9.971457291195739 50.56352602450412, 9.971320026220566 50.56375095378895, 9.971330104034884 50.56376431779629, 9.972078721452041 50.56336139861041, 9.97207645849998 50.56334626646215, 9.971648273975568 50.56316203409465, 9.971655425283483 50.56313992629092, 9.972480266741737 50.562607972164415, 9.97290525597892 50.562824279495096, 9.973145961940164 50.56265240777497, 9.973506683702151 50.562425050312044, 9.973891699231201 50.56221442349825, 9.97430211680075 50.56201996186402)), ((9.977979353247772 50.562351069896955, 9.977346885506353 50.56222832655374, 9.97722887677242 50.56225276257607, 9.977562599688 50.56314036946565, 9.977574844476674 50.563250841919235, 9.978523249379313 50.563073881194434, 9.979932833612976 50.56270264580207, 9.979924472477993 50.5626916050408, 9.9790689210526 50.562631187632974, 9.978520668448299 50.56247865441347, 9.977979353247772 50.562351069896955)), ((9.970507340772913 50.56329373653776, 9.97053581865368 50.563255894484314, 9.970285014586029 50.56277423587327, 9.97017925171609 50.56277918351794, 9.970063237061051 50.562839867633095, 9.969973991532653 50.563001973238705, 9.969952216336495 50.563099692591116, 9.969956681348929 50.563155994197565, 9.970064362827879 50.56314105325957, 9.97023039512759 50.56317499776519, 9.97038935285207 50.5632580293187, 9.970507340772913 50.56329373653776)), ((9.973616716721185 50.56293491938758, 9.972400845535587 50.56326684605984, 9.973663903881583 50.56362078782893, 9.973920098156729 50.563469005375836, 9.974240932167094 50.56337029842523, 9.973616716721185 50.56293491938758)), ((9.976767690309643 50.56366989813956, 9.976761298705824 50.56332392036137, 9.975853331087768 50.56335002970871, 9.975143771225119 50.563526579691136, 9.975765489439052 50.56362087362123, 9.976767690309643 50.56366989813956)), ((9.97518272938271 50.5648706948736, 9.975073041660366 50.56514832944929, 9.97530870813118 50.56515694077496, 9.975453236262124 50.565371350226904, 9.975640275010601 50.56557692964905, 9.975864532346186 50.565766610591446, 9.97613179362125 50.56594188935328, 9.97672804660886 50.56576353234418, 9.976830150360597 50.56569556080007, 9.976909600744934 50.56561515430487, 9.97703170215663 50.56524618913156, 9.976842953805157 50.56518461415661, 9.975912846069594 50.56497921835604, 9.975748253674388 50.56495867542621, 9.975472218492783 50.564881593901106, 9.97518272938271 50.5648706948736)), ((9.977090776085122 50.56525385902622, 9.977061583128997 50.56534077406744, 9.977953862518014 50.56580082591954, 9.978569382656065 50.56606868123123, 9.97845926275663 50.565874494058924, 9.97843774105082 50.56574525211041, 9.97844599982225 50.56565499691922, 9.97841771853302 50.56556710624707, 9.978353956781067 50.56548594172836, 9.978259945474889 50.56541883875818, 9.977090776085122 50.56525385902622)), ((9.978049763264064 50.56622553759268, 9.978531392099939 50.566099498993104, 9.977670851348567 50.56570516079258, 9.977711119656842 50.56585529049936, 9.977791118409941 50.56599053867156, 9.977905692893374 50.56611561960534, 9.978049763264064 50.56622553759268)), ((9.975473495981104 50.56618892803452, 9.975464526207274 50.56617396143788, 9.974765854600395 50.56639368799942, 9.974913471619704 50.56652340798361, 9.975103225540884 50.56645058299338, 9.975271739886509 50.56635819977379, 9.975473495981104 50.56618892803452)), ((9.974184561820842 50.56377050095757, 9.97361016702519 50.56363836948297, 9.972597797122939 50.56335614719498, 9.972458266566882 50.5637656774544, 9.972410684713886 50.56385852446897, 9.972297943296066 50.56397978486146, 9.972108445112024 50.564095167060266, 9.972402031602456 50.564217374220526, 9.972854950993884 50.564485577529844, 9.973100258616656 50.56457688611767, 9.973571169278546 50.564708058395894, 9.974090098247386 50.56478517415886, 9.974184561820842 50.56377050095757)), ((9.976405268856388 50.563918306513834, 9.976367287764482 50.56395865197429, 9.97631803185499 50.56397448485123, 9.97608142128461 50.563972672771804, 9.975639510666053 50.564005072970524, 9.975987029200821 50.564145772091535, 9.976269556034847 50.56421971991144, 9.976565558399793 50.564268572671416, 9.976888654200158 50.5642906296069, 9.977234201060316 50.56428775971996, 9.977745914317898 50.564239478972695, 9.978077992022476 50.56417840783383, 9.97840918569787 50.564086248362266, 9.978283582098577 50.56398807180317, 9.978063669750323 50.56386988158713, 9.977629980581227 50.563720509762334, 9.977569975522755 50.56373892853869, 9.975770673011683 50.563654002319495, 9.975940154778147 50.563730523273776, 9.976340080583098 50.563861081118766, 9.976405268856388 50.563918306513834)), ((9.97128811432818 50.56403040799912, 9.971261977940067 50.56483669190124, 9.971548577536483 50.56437171593745, 9.972024004392532 50.56410289669972, 9.971944934131628 50.56407368281541, 9.971815362576935 50.564053155278344, 9.97128811432818 50.56403040799912)), ((9.975189738904826 50.564832310131564, 9.97519002929148 50.56486166386486, 9.975473196742186 50.56487259504659, 9.97577121312172 50.5649533670718, 9.976048535200933 50.56499751672836, 9.976764588631688 50.56466246319792, 9.977261142624593 50.56459888021324, 9.977586626809774 50.56453907926563, 9.977903089465341 50.564465867922294, 9.978331520722 50.56433905241185, 9.978422076295539 50.56424745033557, 9.978433847112917 50.56412534198883, 9.977929322518579 50.56423732195381, 9.977586639710799 50.564287824504696, 9.977237416417058 50.56431467001402, 9.976885679906635 50.56431754690284, 9.976555184720272 50.56429495258087, 9.976251665033876 50.56424491555239, 9.975961701782827 50.56416908285766, 9.975558689261634 50.564009366246964, 9.975337372014733 50.56441734787349, 9.975189738904826 50.564832310131564)), ((9.939447326644371 50.57094336314715, 9.938775412402855 50.57110477431138, 9.937639964369238 50.571180377139626, 9.93603917771627 50.57119172982526, 9.934165676432443 50.571332717884765, 9.933063803439866 50.57148222660044, 9.933135358302733 50.572136174857384, 9.933238512472096 50.572422262413454, 9.933685880405477 50.57237695840763, 9.935172473036815 50.57216524840623, 9.935443763159512 50.57215926831175, 9.936045990246773 50.572193568658264, 9.937480621953696 50.57206383231953, 9.938055304526653 50.57198225155408, 9.9386210752085 50.57187799450621, 9.939628662051344 50.571633759857754, 9.940805629921238 50.5714095014707, 9.941102228374989 50.57137611155608, 9.941403292285473 50.571369960930724, 9.94188260686422 50.57141572177605, 9.942501126000463 50.57104269386843, 9.94266506202035 50.57091907929862, 9.942038001262922 50.57082083907062, 9.941158512293308 50.57075539079162, 9.940629073543995 50.57075557272406, 9.9396108070377 50.57090412827898, 9.939447326644371 50.57094336314715)), ((9.983106631141922 50.571044266051935, 9.983007946292664 50.571214732864085, 9.985000852269474 50.57149932760112, 9.985155768981963 50.57144924825462, 9.985146002714867 50.57130472882894, 9.985061565137627 50.57093910771231, 9.985038266493488 50.570659673896785, 9.984986847092507 50.57032445612595, 9.984854913487649 50.56981448370999, 9.984838769465524 50.56965881364176, 9.984627781388067 50.56976616041305, 9.984338239950056 50.56999019174622, 9.98414069633565 50.57011183686967, 9.983807188854874 50.570266201199715, 9.983591106653071 50.570331952973575, 9.983472199603643 50.57040302674597, 9.983385100149661 50.57048940523433, 9.983325600364996 50.57059172096195, 9.983197694004698 50.570904309134335, 9.983106631141922 50.571044266051935)), ((9.978476762153017 50.57123029331077, 9.978486981217433 50.571244856546066, 9.979261713221925 50.570884338166316, 9.980181562160487 50.57040028106075, 9.980232096136461 50.57007741718491, 9.979805104202665 50.5702691079437, 9.979469474141695 50.57037677753351, 9.979068506274173 50.570547525379986, 9.978940306378144 50.570640816089856, 9.978593605003766 50.57111825032249, 9.978476762153017 50.57123029331077)), ((9.980174523278954 50.570471755791075, 9.980161275759864 50.570462456764695, 9.979303551642035 50.57091187916112, 9.97844993699312 50.57131140168246, 9.978176895947902 50.571453955249375, 9.97797711727089 50.571593148835476, 9.978305715614354 50.572193906032304, 9.978477216605794 50.57260753008208, 9.978621772164589 50.57304772801429, 9.978750400842884 50.57365430803453, 9.97910709344816 50.573487368415556, 9.980521776050827 50.572981342347745, 9.980444360573918 50.572824343314146, 9.980433480087237 50.572458582802376, 9.980387870181605 50.57220099539144, 9.980348430040065 50.5720731191123, 9.980238510938124 50.57182536634925, 9.980177775308865 50.57154479946009, 9.980137058620047 50.57097962838765, 9.980145060822027 50.57070109913799, 9.980174523278954 50.570471755791075)), ((9.984903537839369 50.571531910222355, 9.984902017508317 50.571514439610986, 9.982986046126905 50.571239328522665, 9.982903343447331 50.57136251818456, 9.982798520800614 50.57156952438733, 9.983040123663432 50.572136783845764, 9.984903537839369 50.571531910222355)), ((9.985178179707416 50.571495842405206, 9.985166437660162 50.57148812466089, 9.982889499804072 50.57223746874995, 9.984130092736136 50.57321101956078, 9.984403116250332 50.57336535811326, 9.984721588745861 50.57349420903818, 9.985013895924892 50.5735769640338, 9.985411957846088 50.57364903994007, 9.985851322811211 50.57369056124211, 9.986080584135456 50.573743298979934, 9.986078512417143 50.5737054920999, 9.985896509281199 50.573431812289364, 9.985737843500877 50.573128424335586, 9.985658141007466 50.57302940481253, 9.985351964335477 50.57232571202571, 9.985194805034462 50.57167284148503, 9.985178179707416 50.571495842405206)), ((9.977981447090729 50.571687539935006, 9.977928975223778 50.57161876797665, 9.977319165623422 50.57189013789044, 9.977083576928823 50.572065535919755, 9.97693894950185 50.57210796412574, 9.976812470641823 50.572750719680435, 9.977333466600475 50.57415863922514, 9.977411224343248 50.57449037660274, 9.97743669747388 50.57451578275011, 9.977447531019616 50.57450723093132, 9.97742441258384 50.57436774073203, 9.97742283555461 50.57411480822967, 9.977611286063771 50.574086601002755, 9.978280040396815 50.57379117091145, 9.978605074895027 50.57372024012293, 9.97870112981966 50.573674607320854, 9.978581346456593 50.57307751250805, 9.97843826885441 50.57263594436162, 9.978192239333572 50.5720620679232, 9.977981447090729 50.571687539935006)), ((9.936984787775982 50.57215108529123, 9.936477115190065 50.5721814329351, 9.93605556497303 50.57222943506053, 9.935744548982441 50.57222240650008, 9.935434740156506 50.57219505271004, 9.935175548026553 50.57220125777221, 9.933697635342915 50.57241233147602, 9.933252887695167 50.5724575603602, 9.933301708719089 50.57263213093513, 9.933344420941689 50.57287598646952, 9.933140136022613 50.573327552200475, 9.93404718152879 50.573484890732786, 9.935366567135974 50.57367777176828, 9.935524002794242 50.573713019595665, 9.940229794787117 50.572461662447374, 9.940323671404508 50.57234293040412, 9.940800723479027 50.5720819948837, 9.941660940009156 50.571657498020656, 9.941661530512674 50.57164369136777, 9.941572446205166 50.57158964114427, 9.941810182694075 50.57145041960306, 9.941688868951188 50.57142629997406, 9.941399227282036 50.571405851391546, 9.941108381033905 50.57141191188285, 9.940821447465746 50.571444375626236, 9.939651719165871 50.57166733975982, 9.938646941388415 50.57191099441843, 9.938362205455451 50.5719666072804, 9.93778486176024 50.572060736330954, 9.936984787775982 50.57215108529123)), ((9.980915365197902 50.57289219008711, 9.980706719238553 50.57306212379094, 9.981127480067764 50.5732012739176, 9.981378770248991 50.57332096261751, 9.982189816618188 50.573838530395605, 9.982693909995227 50.574126271811394, 9.983190901949023 50.574383687143914, 9.983682222861328 50.574610802914606, 9.984481867531102 50.574920580173675, 9.985203413994203 50.575142555416036, 9.985846607433768 50.57529676199492, 9.98586924345443 50.57512142760956, 9.985972317374307 50.574785463023396, 9.986093678896875 50.5746564165487, 9.986133841776462 50.57443913328, 9.985449643446211 50.574270680338074, 9.98528240849889 50.57413182022173, 9.98321006343654 50.57329915483564, 9.98323220159209 50.57327519811706, 9.985326961333916 50.57411594281408, 9.985490420016491 50.57425157896846, 9.986138694319385 50.574411788387124, 9.98622510251406 50.57402127197295, 9.9860913374223 50.573774074842184, 9.985833238058612 50.57371218198213, 9.985390423036442 50.573674024538896, 9.985010912673923 50.573607851561356, 9.984719416679981 50.573525191848994, 9.984384102166496 50.57339153412727, 9.98408849828184 50.57322546551397, 9.982836644817768 50.57224591953173, 9.980915365197902 50.57289219008711)), ((9.980845483900003 50.57293461385186, 9.980837175451374 50.572918896526055, 9.9806095357479 50.57300634820755, 9.980680662672253 50.573057173485644, 9.980845483900003 50.57293461385186)), ((9.940177908265737 50.57253201106361, 9.940168467401648 50.57251781845886, 9.936105943523804 50.57360114227539, 9.935974053315569 50.57386217621759, 9.936286253886987 50.57418467550195, 9.937222644126397 50.575457341706915, 9.937796515284933 50.576266505835484, 9.937810184488816 50.57626027509833, 9.937843428518507 50.57604848851024, 9.937876146589934 50.57552244676399, 9.937904946864263 50.575322416415915, 9.93797264644308 50.575126094490265, 9.938078274461628 50.57493648003316, 9.938949577363825 50.57375674243168, 9.939145154142004 50.573542443300795, 9.93970940418279 50.57311519581405, 9.940177908265737 50.57253201106361)), ((9.980635294293837 50.573092230242636, 9.980636014399721 50.57307843044287, 9.980552913592415 50.57301598812651, 9.979492472698075 50.57338705227434, 9.980248902278637 50.57329238851774, 9.980418336680879 50.57323613146485, 9.980635294293837 50.573092230242636)), ((9.935392251887109 50.573748050854164, 9.935393800197296 50.573731247014884, 9.935342774997524 50.573718361061694, 9.933109620410956 50.573387973786424, 9.933126286533986 50.5739036049826, 9.933192149445976 50.5743410193269, 9.935392251887109 50.573748050854164)), ((9.981043224013606 50.57320217862073, 9.98066347773718 50.57308423995777, 9.980432623308927 50.57324096771375, 9.98034286626416 50.57327468830363, 9.97942339371783 50.57341447157821, 9.979118132840846 50.57349406091894, 9.978706907202282 50.57370565909513, 9.97829356021946 50.57380152532969, 9.978287022218286 50.573867757530934, 9.978531564674558 50.57402247546169, 9.978936468580455 50.574175276707884, 9.979374879460426 50.57430918686301, 9.980097823013702 50.574482818239176, 9.980372112253981 50.574585963773295, 9.980533486715611 50.57471687198635, 9.980691770866224 50.574996586657186, 9.980892904269073 50.57521739484871, 9.981150061758619 50.57541375438621, 9.981455197713261 50.57557970912926, 9.981728616639579 50.5756877798854, 9.981967249332989 50.57575794781893, 9.982218678511135 50.57580695376847, 9.98256198962339 50.57584182492328, 9.982353911586587 50.57548013754252, 9.98220914701197 50.57513112004658, 9.982074929374678 50.57488325630234, 9.98183473739515 50.57456789250854, 9.981689784950486 50.57441825767122, 9.981454701212163 50.574214800041915, 9.981490535830067 50.57419716441139, 9.98181109265707 50.57448356261805, 9.982071296234066 50.57479696557385, 9.982220470522261 50.57504403052038, 9.982458817944416 50.57557934248488, 9.982635908039112 50.575853129082674, 9.983456258037089 50.5760210396987, 9.983993207322605 50.57610929801234, 9.9842222539553 50.57620425554553, 9.985925879511868 50.57667836608681, 9.985975055851224 50.57636244948155, 9.985862524922801 50.575884593113976, 9.985825851836019 50.575466965004566, 9.98583823613472 50.57533206835266, 9.985255938099149 50.57518766761588, 9.984666435262264 50.575013796370435, 9.984002217179313 50.574777046386714, 9.983590544066125 50.574606637376654, 9.982677192690863 50.574156145755, 9.981902366186743 50.573705439761554, 9.981343801067338 50.573340591629616, 9.981043224013606 50.57320217862073)), ((9.936035372428844 50.57363250751304, 9.936025089910098 50.573619628594294, 9.935628137990268 50.57373494973603, 9.935936246214775 50.57381607685194, 9.936035372428844 50.57363250751304)), ((9.939019639160154 50.573769237344386, 9.938621289790685 50.57429967551133, 9.938700304873219 50.574831659091544, 9.93873031228464 50.574915324713224, 9.938881169355598 50.57515085022209, 9.93903268385203 50.575291467653024, 9.939041165475947 50.575280876030085, 9.939022212587089 50.575231332244414, 9.939429781033462 50.57436181045981, 9.939640942395888 50.574002291098715, 9.939019639160154 50.573769237344386)), ((9.980453492007767 50.565648848644095, 9.980453100909706 50.565666159548265, 9.980697001310373 50.56569085828759, 9.980986578659445 50.5657585749251, 9.981566045124705 50.56596279873065, 9.982305527355084 50.566261993931235, 9.983470857156977 50.56662038317683, 9.983846820699291 50.5667813757874, 9.98390490253686 50.56678784620842, 9.983013035509197 50.566244904262796, 9.982218542706619 50.56569555365877, 9.981994891618466 50.56547565264027, 9.981876670832422 50.56528089772751, 9.980453492007767 50.565648848644095)), ((9.98407557977386 50.56685544727636, 9.983818539624375 50.56678538006121, 9.98357276747747 50.56668760716149, 9.983306415047364 50.566876130468515, 9.983376138596173 50.56692503887562, 9.983400601613106 50.56723852590695, 9.983369573502568 50.56729098151028, 9.983310998095279 50.567334655721424, 9.983186944815483 50.56737092564097, 9.9830949730546 50.56737076401625, 9.982011452524915 50.56715924577559, 9.9820008143637 50.567169381421806, 9.982256891698762 50.56731651423571, 9.982797293653505 50.567578324628435, 9.983833961085065 50.567946180450406, 9.984533574599014 50.56808514064798, 9.984258574340366 50.56692741361869, 9.98407557977386 50.56685544727636)), ((9.98009507695967 50.5667529810669, 9.980096636311494 50.56673619450269, 9.979958798410985 50.56670103184625, 9.979677007978497 50.566647522964004, 9.97938762959536 50.56661578517073, 9.978900103321541 50.56660539836045, 9.978621569898122 50.56655401463719, 9.97838480552472 50.56644805187966, 9.978263224006767 50.56635282272993, 9.97816950780604 50.566238965014385, 9.97657709724739 50.5666459800505, 9.976628438443004 50.56677537823864, 9.976715537776203 50.56688567111083, 9.976878438449894 50.56702202660157, 9.97690256356612 50.56706626926333, 9.976903464174539 50.56711229683733, 9.976881120651791 50.567156814762335, 9.9768366981452 50.56719557284799, 9.976771757360812 50.56722518794165, 9.97646536681579 50.56745885961097, 9.97635172795843 50.56750615049554, 9.976220110571026 50.5675286221866, 9.975865909270386 50.56753058491163, 9.975548500603574 50.56757852965228, 9.975408727040568 50.567558418932904, 9.975334214736652 50.5675100447609, 9.975229927602069 50.56736611575377, 9.97517426741829 50.567236402292146, 9.97514077055065 50.56709461697385, 9.973903519663875 50.56788871409875, 9.97302244042055 50.56853009236852, 9.973027804954352 50.5688848885089, 9.973051514735781 50.56908139373839, 9.973139066227425 50.56931373136205, 9.973496568706937 50.569279299427606, 9.973523276358602 50.569296578640554, 9.973855927491153 50.569067900265594, 9.974058972675541 50.56896696224205, 9.974626313743306 50.56870132373402, 9.975261429764245 50.568476545683744, 9.975263315048261 50.56846082324005, 9.975157678854117 50.56840637044204, 9.975081514167874 50.56833747226155, 9.975038381383063 50.568258817315716, 9.97503078684685 50.56817772737939, 9.975124030764835 50.56807037317163, 9.97528340315519 50.56799623387748, 9.975488112502587 50.56796950595822, 9.975698094693298 50.56799619563519, 9.975874248346514 50.568072831757284, 9.975997545353115 50.56819033161553, 9.976137719907898 50.56813499702369, 9.976640825680029 50.56798712540845, 9.977913045276608 50.567680152057704, 9.97810165127915 50.567618195566894, 9.978454080635897 50.56744067147977, 9.978318640311972 50.56734864691902, 9.978057293556718 50.5672329605696, 9.978003528119935 50.56719057205443, 9.97796838800041 50.56711463459172, 9.97797897001671 50.56701836592979, 9.97801945204341 50.56696499067287, 9.97806920092928 50.566938629702776, 9.978532214362009 50.5668988955851, 9.979017580403761 50.566893736726286, 9.979081148375966 50.56691260143266, 9.979121200501265 50.56694625147456, 9.979167747792998 50.56709179628999, 9.979614588025632 50.56690090511822, 9.98009507695967 50.5667529810669)), ((9.978469353925542 50.567455145135284, 9.978121590357013 50.56762378886366, 9.977936083300369 50.56768673461261, 9.976898630629652 50.567934097088035, 9.976161231590535 50.56813868787318, 9.975063932007894 50.568564220719296, 9.974635416380826 50.5687115705802, 9.973876641713364 50.56906986060721, 9.973537683373312 50.56931149318228, 9.973850191421427 50.569553394165005, 9.974109037946466 50.569955076710514, 9.974288662384598 50.57039449986383, 9.974569532994302 50.57060947350827, 9.974889024879259 50.5703275741979, 9.977597165085736 50.56902014156842, 9.97815943293464 50.56850313571686, 9.978986764143006 50.56785370556258, 9.9789515409142 50.567766090499426, 9.97892953245514 50.567596370251785, 9.9790999298462 50.56746767920842, 9.979404507576646 50.56716047445571, 9.979512089229358 50.56696982595087, 9.97897075758206 50.56718206379071, 9.978469353925542 50.567455145135284)), ((9.97237445906778 50.56423262729537, 9.972281339178684 50.5641825906649, 9.972082471924562 50.564112024621124, 9.971592292973497 50.564382691095346, 9.97120066580702 50.565035349026644, 9.970788067112462 50.56595358760542, 9.970323905954386 50.56712502586235, 9.969899117169412 50.568734044564046, 9.970221841437905 50.56882951988495, 9.97074725078093 50.56888480742376, 9.971355475201015 50.56888452422404, 9.972227226536784 50.568783236325125, 9.972894171230266 50.5685639741511, 9.973852680338279 50.56786607830364, 9.974998455390882 50.56712312930673, 9.974886426159742 50.567010188934496, 9.974640375946914 50.56682042553139, 9.974345413651147 50.5666477559862, 9.974278174323558 50.56657263053519, 9.974200577744268 50.566419829045245, 9.973985536882097 50.56626554607824, 9.973816922560689 50.56611934224593, 9.973629080590014 50.565871059605534, 9.97304002667821 50.56589741903687, 9.972842458862457 50.565987319552704, 9.972742072712535 50.56606801187558, 9.972685406804942 50.56614037023434, 9.97237445906778 50.56423262729537)), ((9.984175728205063 50.56389143440003, 9.982056685720796 50.5639887650991, 9.981977894091301 50.56424022814155, 9.981915210859876 50.56453039006123, 9.981881700689444 50.56497042567712, 9.981899459472977 50.56514036712486, 9.981958586189545 50.56530623025592, 9.982057465490648 50.5654638807925, 9.98227393786285 50.56567626977158, 9.982824498223334 50.566065228961044, 9.98367890885766 50.56660452790302, 9.98393476325353 50.566746710929685, 9.984250104061664 50.566878506021496, 9.984258344293046 50.56686737904933, 9.984196739545911 50.56661584276021, 9.984132417661034 50.56559591786666, 9.984109760257116 50.56445309481383, 9.98416951373721 50.56425925067421, 9.984175728205063 50.56389143440003)), ((9.971232431396636 50.56405368660475, 9.971218746964464 50.564045682884455, 9.970966322144923 50.56419979053656, 9.970683797362325 50.564438727321885, 9.970426461308554 50.56494699270462, 9.970664656341919 50.56499138709404, 9.970818498306764 50.56499972687218, 9.970674587223149 50.56545407854603, 9.970517201663872 50.565794518044974, 9.970267395544193 50.56621071855642, 9.969367095995567 50.567424220114184, 9.969830899379057 50.56760314194021, 9.969459033709711 50.56864069155106, 9.969844559203294 50.568712093498135, 9.970271707468964 50.56712187044491, 9.970736549700531 50.56594871763529, 9.971149345450145 50.56502990678922, 9.971199941513845 50.56494574308429, 9.971232431396636 50.56405368660475)), ((9.98051116577419 50.56667688217187, 9.980507027710216 50.56669387118925, 9.980829356055372 50.56686843071719, 9.981266271067902 50.56716102186672, 9.981523593946674 50.56737459841505, 9.981740742057408 50.56758990866116, 9.981405508757977 50.56767219090581, 9.981454129756683 50.56878057385697, 9.980860918539953 50.56920461097999, 9.98067507495046 50.5693874565417, 9.980518623583547 50.569582518784635, 9.980431542850372 50.56971865408454, 9.98032820370939 50.56993809750029, 9.980313866293347 50.57001211990341, 9.980658658758228 50.56980883052093, 9.980799450330649 50.56968686935433, 9.98106455716445 50.569518200009114, 9.981373025214877 50.56927525286644, 9.981708941660253 50.56893037937796, 9.982063432448392 50.56866791219543, 9.982232731504654 50.568473759352976, 9.982342417007748 50.56823476740634, 9.98238579906775 50.5681759450624, 9.98246185328586 50.56812935577875, 9.982551886141096 50.56810322870489, 9.982755092612928 50.56810748722559, 9.982742318252914 50.56797939375636, 9.98275481610159 50.56786973267162, 9.98279099802136 50.56776081078138, 9.982862598274492 50.56762049546442, 9.982240628576589 50.56732766136996, 9.981870270578742 50.56712344182886, 9.981002417166186 50.56673919060072, 9.98060003106448 50.56667692593235, 9.98051116577419 50.56667688217187)), ((9.981355952551887 50.567687712110335, 9.981345130319463 50.56767930785301, 9.980369266415561 50.56785960856183, 9.97965680549192 50.56795523057761, 9.979417175361156 50.56816644755347, 9.97916979844806 50.56801786071352, 9.978993525557382 50.567868046887135, 9.97817789193618 50.56850684318013, 9.977719764147762 50.56892747898699, 9.977215232670847 50.569420703612806, 9.976906060967572 50.56975682982314, 9.976371847815011 50.57018844617261, 9.976327592980022 50.570210755222774, 9.976818401854336 50.56981925205429, 9.977509087100966 50.56909322160476, 9.97493696276375 50.57033834341988, 9.974604298541722 50.570639904978854, 9.974878254099513 50.57085582189902, 9.975165290268201 50.5710065963452, 9.976380855360127 50.57020416626479, 9.976548108297532 50.57015064907432, 9.976722292091706 50.57017576351492, 9.976807787036206 50.57026325496981, 9.976861281097175 50.570554119235005, 9.977074169799515 50.571126542365384, 9.977045389638828 50.571552165957385, 9.976942641487456 50.572081818986526, 9.977023373504911 50.572074801433544, 9.977300439629927 50.571882934333274, 9.97792103261844 50.57160762166214, 9.978397042379951 50.57127907835205, 9.97856350242296 50.57111774980956, 9.978764238350141 50.57086191801537, 9.97883761003796 50.57073323131208, 9.978949441311427 50.57061235179772, 9.979178914128308 50.57048463985342, 9.979789697777017 50.570260651054724, 9.980240844763284 50.570050975388774, 9.980305599332755 50.56986969724589, 9.980405459252024 50.56967579135295, 9.98063552856756 50.569369023156554, 9.980819027720134 50.56919009128701, 9.98140427668591 50.56876964145027, 9.981355952551887 50.567687712110335), (9.976729471530952 50.57015876264411, 9.976569133522403 50.57010471377522, 9.97769648997155 50.56917343976582, 9.978562041757106 50.56872088030628, 9.97886433223724 50.56861508036676, 9.979061838766537 50.568714881531264, 9.97870708223373 50.56901799879169, 9.978236529846972 50.56937522508932, 9.977502532581711 50.56986066778837, 9.977023195075242 50.57025778387092, 9.976729471530952 50.57015876264411)), ((9.983103923775671 50.57027271451677, 9.983242975597733 50.570149554699555, 9.983324312851284 50.57002103109176, 9.983598701436751 50.569749702190116, 9.984161281322384 50.56914801682149, 9.984473549932176 50.56874667515024, 9.984572826623499 50.5685293318779, 9.984524620113145 50.56831706211866, 9.984529357421227 50.56822301391902, 9.984566446777082 50.568116688968374, 9.984520783582626 50.56810756044174, 9.983885822318955 50.56840699188298, 9.983496329985263 50.56863295134119, 9.982225101915306 50.56929313484711, 9.981631587268321 50.56965033594316, 9.980236653613566 50.57042136729087, 9.980189976198199 50.57084154136124, 9.980211172073734 50.571400339696254, 9.980289874394916 50.57181941136612, 9.980401184742707 50.572071639854684, 9.980440357178532 50.572199129625716, 9.980485999847383 50.57245714536287, 9.980496139909738 50.572819726738324, 9.980569520844742 50.57296293147545, 9.980958177998964 50.57283444512919, 9.981231049239748 50.572416402459154, 9.981382212015449 50.572252330849416, 9.981899135010948 50.57177390870669, 9.98210002387803 50.57146564151467, 9.982348896608242 50.57103413145256, 9.982528384760313 50.5707946109106, 9.982738406134278 50.57056811583078, 9.983103923775671 50.57027271451677)), ((9.983106664013404 50.5702917599601, 9.982754890212346 50.57057303214132, 9.982548411378858 50.57080042559322, 9.982363370134674 50.57104093353128, 9.982124574051909 50.57146582486443, 9.981916043298325 50.571776363469795, 9.981397292942832 50.57225681255331, 9.981261864380427 50.5724168128265, 9.980997127234572 50.572817985606065, 9.982985398304884 50.572153454072954, 9.98274564293112 50.57156752615573, 9.982857960563244 50.57134755666833, 9.983145362970193 50.57090023232651, 9.98322455407891 50.570687068197984, 9.983334498722702 50.57048256944891, 9.98342634210619 50.570390508500566, 9.983551110501866 50.57031475310342, 9.983922059777832 50.570183981099, 9.98417675092885 50.57005140809562, 9.984589777114037 50.56974814165007, 9.984830413231471 50.56961523045837, 9.984681446844782 50.56884144859668, 9.98460312946708 50.56833951946033, 9.984568134118554 50.56821918493993, 9.984552408141484 50.568230045513374, 9.984553943839698 50.568320209141454, 9.984593499455803 50.56852184330332, 9.98458692534071 50.568568820676354, 9.984493653618973 50.568749079744286, 9.984185568351752 50.56915028673537, 9.983623459249186 50.56975293101846, 9.983358339349136 50.57001494329334, 9.983193139527232 50.57022410184775, 9.983106664013404 50.5702917599601)), ((9.982700003024508 50.56899928628031, 9.982740675913607 50.568648445682975, 9.982841402534744 50.5682717180057, 9.982760491358201 50.56812020510025, 9.982563232194186 50.568116272706256, 9.982478108488538 50.5681354146808, 9.982400480696926 50.568188685179706, 9.9822532349613 50.56847051923327, 9.982175683314514 50.56857734798397, 9.982082933022296 50.56867111022304, 9.98173549922643 50.568932445309976, 9.981586288516347 50.56906959651079, 9.981398795074494 50.56927591183152, 9.981073678045968 50.569525855084116, 9.980824679364263 50.56969112326098, 9.98067713584722 50.569813868747914, 9.980462292034522 50.569951111252635, 9.980297324065315 50.570034941115686, 9.980235005301495 50.5703600352918, 9.981279623602399 50.56979786697356, 9.982177255042929 50.56926862113508, 9.982700003024508 50.56899928628031)), ((9.969666302437078 50.56035866149917, 9.969946493709864 50.56040157572518, 9.970177575999429 50.560457023356506, 9.97048971682108 50.56058308564254, 9.970666644598268 50.56069234469491, 9.97081299719618 50.560818329897614, 9.970924901052593 50.56095755433382, 9.971015742513615 50.5611401417147, 9.971285625268871 50.5611924122633, 9.971196887063034 50.56099167630233, 9.971079254606453 50.56083935162375, 9.970926118933013 50.560700121325965, 9.970742756037836 50.56057842185293, 9.969808939550381 50.56013559382505, 9.96831591164013 50.55946449584356, 9.968038743568034 50.55976534105503, 9.968114619777879 50.55983953952161, 9.968286950786315 50.559954619952485, 9.968488709915633 50.56004796009793, 9.968713355705166 50.56011660334536, 9.968954836515394 50.56015838329161, 9.96899233727958 50.56008397926665, 9.969071605147231 50.56003409953719, 9.969141651734867 50.56001782075503, 9.969278024803229 50.560058031267296, 9.969444427231236 50.56015550312721, 9.969545214993769 50.560286960924, 9.969666302437078 50.56035866149917)), ((9.972743544989449 50.56017312392842, 9.972928062056683 50.559876538988156, 9.971862379889203 50.55917975897333, 9.971399249170885 50.5594228849135, 9.971009281452526 50.55971167725527, 9.970701239299535 50.560038150191815, 9.97054600760805 50.56027176554761, 9.970483017709853 50.56039793937545, 9.970790300003216 50.56055383647479, 9.971075299239063 50.560752816175196, 9.971321562467983 50.56054591093136, 9.971494353059823 50.56031108874931, 9.971565546644257 50.56014279287824, 9.971598920158556 50.55997141123658, 9.971589230424515 50.559877413876016, 9.97164699751187 50.559747002674186, 9.971730225738082 50.55967204915795, 9.971888699925577 50.55960787029933, 9.971974220176811 50.5595256340196, 9.972023250880797 50.55951589224631, 9.972130241996659 50.55952706717045, 9.97225190501709 50.55960239890202, 9.97229991815218 50.55966806861902, 9.972325163164859 50.55983311028404, 9.97238387594636 50.55994000311072, 9.972533998578843 50.56007606460406, 9.972743544989449 50.56017312392842)), ((9.965974612798401 50.560245718382426, 9.96596039158045 50.560237500137745, 9.965742980802805 50.56039135078537, 9.965763884297521 50.56063899138011, 9.96582153974286 50.56085014771364, 9.965874751623192 50.560970158862936, 9.966063045251472 50.56096239983392, 9.96597380889208 50.56053509534861, 9.965974612798401 50.560245718382426)), ((9.966401165496487 50.55934645156704, 9.966444272005974 50.55919842662098, 9.966447987908133 50.559006676483264, 9.966486352596963 50.55894032289075, 9.9665541822475 50.55888338773166, 9.966830231932908 50.55875404018676, 9.966251546764523 50.558360899319226, 9.965779768003053 50.55800677898306, 9.965707776653224 50.558231303704346, 9.96568400079518 50.55845632425977, 9.965704956478753 50.55868057210887, 9.96578447590424 50.55893303199483, 9.965572456524225 50.55892571949401, 9.965352587350605 50.558892401837014, 9.965044403624661 50.5587959167699, 9.965029450881264 50.55906424453687, 9.965328700261862 50.55911766461113, 9.965720592962635 50.55913393855742, 9.96581400416082 50.55944275030001, 9.965837511383748 50.55966457845842, 9.965825859569396 50.55988677703997, 9.965756561701681 50.560208669456685, 9.965748272942802 50.56033531520968, 9.965933101217518 50.56021158410925, 9.96598647045071 50.56015272020981, 9.966083985247352 50.55963873416069, 9.966091501560763 50.55955444597275, 9.9660742164497 50.559396520681055, 9.966401165496487 50.55934645156704)), ((9.981278585385134 50.55937319382961, 9.981288076363649 50.55936173192535, 9.98124779790216 50.55923818269356, 9.981087303863472 50.55901359604253, 9.981035551659227 50.55886450832654, 9.981030263351444 50.55881528824388, 9.98112758499298 50.55847081718023, 9.981002510077047 50.55848660187278, 9.980716934270644 50.55848101038824, 9.98032170061371 50.55862087957122, 9.980242682783143 50.55862766794983, 9.980058823990284 50.55873527920043, 9.979897171037688 50.558773724017094, 9.97931157936555 50.55880286911765, 9.979061154120853 50.558758297133316, 9.978590758403994 50.55884210201031, 9.978202483611483 50.55894612295745, 9.977670646036032 50.55918204703694, 9.977736156163969 50.55975252273878, 9.977720921528746 50.560059024220166, 9.977734552709078 50.5602044743996, 9.977750549394974 50.560206986039645, 9.97803470471676 50.55985073802469, 9.978437547580237 50.55947967262344, 9.978655832426972 50.559351292774714, 9.97883188375566 50.55929239108544, 9.97912252025983 50.55925065598824, 9.979673600238637 50.55924825163073, 9.980123017331746 50.55927133850065, 9.981278585385134 50.55937319382961)), ((9.965804493512158 50.560972289840365, 9.965813077704865 50.56096157050515, 9.965711515691345 50.56064116010576, 9.965683801462946 50.56043066899485, 9.965342331533748 50.56061923413483, 9.964955706134349 50.56079946388638, 9.965054388133522 50.56102618498065, 9.965804493512158 50.560972289840365)), ((9.981438019338256 50.56118102447793, 9.981284209594612 50.56104373431658, 9.98081763442889 50.56094968432235, 9.98000692709124 50.56089212850668, 9.979203761667105 50.560766972162284, 9.97913569747148 50.56089874004245, 9.979042536847183 50.561016970152195, 9.978920391739099 50.561124014697455, 9.978703944041245 50.56124901706567, 9.978020949917521 50.56160085888183, 9.97673300628604 50.562182087762864, 9.977281353577403 50.562211001703616, 9.981271861176138 50.5615309422856, 9.981341060145327 50.561334016471456, 9.981438019338256 50.56118102447793)), ((9.974369199096513 50.56055276801707, 9.974367389640637 50.560537339015156, 9.973410822444594 50.560119997196765, 9.973434034021572 50.56027938799759, 9.973603649973045 50.56059292422291, 9.973610287038923 50.560700166218346, 9.97354388507213 50.56084910946556, 9.972957744685651 50.56059222708003, 9.972664725321271 50.56042476094854, 9.972292877362637 50.560969024794815, 9.973087049377556 50.561526261763305, 9.974369199096513 50.56055276801707)), ((9.970728972248274 50.56111307108688, 9.970643146062473 50.56086962140277, 9.970606561011726 50.560805376177676, 9.970537901408548 50.56075182446264, 9.970414741522816 50.560690725323525, 9.970234293937825 50.56062544585456, 9.970037034487024 50.560586624239335, 9.969786587462217 50.56057511856557, 9.969714415486203 50.56061659411433, 9.969713400183496 50.56066245840693, 9.969804972663038 50.560763874781486, 9.969837786006979 50.5608467509083, 9.969814805931193 50.561043241127145, 9.969835667187613 50.561179644819326, 9.969896342403894 50.561309302100334, 9.969995043220202 50.56142937078686, 9.970140370182122 50.561522962344455, 9.97040684776282 50.561621193348394, 9.970606667981416 50.56165476619458, 9.970897048917104 50.56165429403729, 9.970757679384278 50.5618283504311, 9.97073971476588 50.561871027335144, 9.970734685483649 50.56196204956477, 9.970772918094255 50.562049385763714, 9.970644129363057 50.562031916119444, 9.970534440686082 50.56206938659346, 9.970498197272931 50.56210562093735, 9.970417712502314 50.5622849030744, 9.970234614216476 50.56224879980107, 9.970029505276164 50.56226708244506, 9.970089241994536 50.56232180596914, 9.970570332805956 50.56320825530445, 9.970872644958629 50.562827754100546, 9.971437463272004 50.562214241147984, 9.97132836394026 50.5616415150101, 9.971300216559023 50.56123410635557, 9.970728972248274 50.56111307108688)), ((9.968345949439657 50.560305695135725, 9.968084620275613 50.56032372512293, 9.96800386185968 50.560344616218615, 9.967934484154517 50.56040680455927, 9.967930030409196 50.5604692982413, 9.967953483627207 50.560507146847335, 9.968135077549347 50.56060221915114, 9.96822381867825 50.56068532644631, 9.968473336056295 50.560856121613575, 9.968745469261284 50.561012190348656, 9.96912642635446 50.56119463917792, 9.96935172710552 50.561348811779446, 9.969505251859932 50.56153451843756, 9.969610935552236 50.561772043127576, 9.9697349578493 50.56196738627066, 9.969895750498349 50.56215210724582, 9.96995377877004 50.56219649987786, 9.969929373796868 50.56202818415989, 9.96993508793768 50.56186691706058, 9.970007678816462 50.56162370871152, 9.96983781246438 50.5613910752658, 9.969631092677336 50.56117494019798, 9.969388637875904 50.560974597335075, 9.96911373696261 50.56079263684012, 9.96866511719814 50.560533079518635, 9.968345949439657 50.560305695135725)), ((9.966067860968296 50.561543475756494, 9.966517595689234 50.56170748185359, 9.966468108431107 50.5612163709614, 9.96642326254156 50.56096826453221, 9.966050496466288 50.56098601181234, 9.966070555581704 50.56123891026666, 9.966067860968296 50.561543475756494)), ((9.983899633157346 50.561124745751755, 9.983889130147485 50.56111590454378, 9.983075933192687 50.56125517375346, 9.983126420608977 50.56149538542681, 9.983108033871458 50.56165004729997, 9.983082883710635 50.561726091904895, 9.983001503910353 50.561872764050854, 9.98289264906087 50.56200003380535, 9.98268337971688 50.56214904210186, 9.982413742987472 50.56225251779781, 9.981835087922708 50.56234540527441, 9.98156876299191 50.5624164347857, 9.981999980546021 50.563084065140806, 9.982038154850711 50.563194327683604, 9.982099600979371 50.56353217691651, 9.982076690947594 50.56391348148363, 9.9841757346064 50.56380649394466, 9.984208508930156 50.56359107615973, 9.984479042360947 50.563235231462, 9.984506834702376 50.563173650076486, 9.984522786727487 50.56302908132557, 9.984464665793357 50.56280856593028, 9.984331921115313 50.56256465723191, 9.984047217463056 50.56182668500625, 9.98396039849493 50.56168930125099, 9.983900872972155 50.561469215805225, 9.983899633157346 50.561124745751755)), ((9.936297327452802 50.574313595034766, 9.936217632076543 50.574203838143134, 9.935865221670184 50.573853722796535, 9.935518599234522 50.573762740857305, 9.933238310472918 50.5743748851828, 9.931285171101052 50.574934654605784, 9.93003794582803 50.57521775833787, 9.930056903943074 50.575256641700584, 9.930803191514908 50.57540400068763, 9.933828368579674 50.57585671965316, 9.93690313346358 50.57639263738699, 9.937519355975642 50.57660484143165, 9.937663982208857 50.57646676922473, 9.937762711924082 50.57633210121275, 9.936297327452802 50.574313595034766)), ((9.978622729422698 50.57510054858572, 9.978633005461854 50.57508954859162, 9.978447380734483 50.57432801275279, 9.978600353042696 50.574087068987055, 9.978283891157494 50.573909619738515, 9.978209285682068 50.57427401831023, 9.978003472613668 50.57463711689079, 9.978133861024764 50.574923861425525, 9.978622729422698 50.57510054858572)), ((9.978703222660851 50.57510919442076, 9.978874222387812 50.57514399340753, 9.979115409869287 50.57515495829313, 9.979447402096477 50.57511666771155, 9.979667807339968 50.57475644060151, 9.980160716685875 50.574535187887335, 9.979318497161005 50.574322570160085, 9.978644445851529 50.57410297894449, 9.978491825962058 50.57433553259173, 9.978703222660851 50.57510919442076)), ((9.979869666956295 50.57583769344877, 9.980014130868119 50.575927972313295, 9.980178343615934 50.57599862367531, 9.980359778560437 50.57604991937137, 9.980551314808059 50.57607999311456, 9.981107623861128 50.57609721583877, 9.981680615241414 50.576066692449245, 9.982891278729001 50.57621460571577, 9.98259596339204 50.57587583899262, 9.982161113527196 50.575825386121686, 9.981862589367564 50.57575955864078, 9.981493167850092 50.57562955401815, 9.981170409683152 50.57546544925329, 9.98094699293455 50.57531102120271, 9.980758973713025 50.57513915952647, 9.980609796462725 50.57495313209104, 9.98055132026461 50.57485706963689, 9.980519619892574 50.57476797417908, 9.980454980813526 50.57468556270732, 9.980361142810878 50.57461524643381, 9.980226128129061 50.574555018623926, 9.979706733016569 50.574775372985336, 9.979507101417383 50.575096813566326, 9.979512958484682 50.575108523002555, 9.979700960659518 50.57513588642216, 9.979808705688596 50.57519911805709, 9.979847948590754 50.57529933812942, 9.97984878446178 50.57580162945687, 9.979869666956295 50.57583769344877)), ((9.938628625808795 50.57470434108414, 9.938571248586438 50.57438277049394, 9.938541178934189 50.574551704625826, 9.938533775156557 50.57485161137325, 9.938554201372593 50.57505193990613, 9.938594770497497 50.575250535582235, 9.938655328065753 50.57544701618955, 9.9387791989259 50.57573842710158, 9.938894091686553 50.57596150521887, 9.939014602442573 50.576086921797575, 9.9391268338926 50.57615529904758, 9.94059785485443 50.57664167471026, 9.941211640881752 50.57668327762262, 9.941207391403577 50.576669006455205, 9.941021452613576 50.57661001797246, 9.940189902592545 50.576009183060584, 9.939074584128377 50.575371677636205, 9.938985076388892 50.57530554789339, 9.93883266978803 50.575160465420154, 9.938719321343426 50.575001896504354, 9.938647984730485 50.57483439164508, 9.938628625808795 50.57470434108414)), ((9.938495583999197 50.5744955170923, 9.938479423392556 50.574488998883645, 9.938141718845255 50.574946941643404, 9.937999479343485 50.57523001903073, 9.937952027359524 50.575425774061394, 9.937935975980519 50.575784674972695, 9.937902118539183 50.576108059766405, 9.937875789192985 50.57626775904282, 9.937841615854767 50.57632704276688, 9.937891913371095 50.57640216470898, 9.938047070854749 50.5765540725503, 9.938284934177865 50.57664123743245, 9.940453795633236 50.57663931181285, 9.940450126437415 50.576624536111495, 9.939094957956248 50.576176464470436, 9.938974423996367 50.57610375100571, 9.938893751752707 50.576032073403205, 9.93883397380004 50.575952928531436, 9.938727768693342 50.57574350351289, 9.938603064270705 50.57544992552646, 9.938541801894178 50.57525049703805, 9.93850102186844 50.57504899268006, 9.938480887302815 50.57484625083389, 9.938495583999197 50.5744955170923)), ((9.934807723482482 50.576058698633446, 9.932689777626488 50.575715121299346, 9.931150162752592 50.575500575741195, 9.93007312029373 50.57530734313873, 9.93028500500645 50.57587191011405, 9.931046233658241 50.576705962611534, 9.931571896756317 50.57711289214385, 9.932670424351539 50.57680861769169, 9.934056219491254 50.5766534287756, 9.937461354568622 50.576637424389276, 9.937202447195986 50.57652803320065, 9.936878295204727 50.576425952881486, 9.934807723482482 50.576058698633446)), ((9.98150376093987 50.57672072888133, 9.982245035579341 50.576804828376744, 9.982476956538507 50.57692402606228, 9.982962792772568 50.577337395630884, 9.983245371270774 50.57750414365059, 9.98336167065446 50.5775989198259, 9.983416476023947 50.5776867173489, 9.983427254457563 50.57777371477841, 9.983563924224544 50.57761456005445, 9.983912901970891 50.57713591361729, 9.983675883958025 50.576943673968586, 9.98334525489189 50.57661448759688, 9.982940004096013 50.57625724249616, 9.981674174597988 50.57609412886809, 9.981109600515946 50.57612415897545, 9.980543580656043 50.576106641121875, 9.98024419462635 50.57605078602563, 9.97993412148295 50.575933044388634, 9.979989146527714 50.576028402854575, 9.9801765407218 50.5761312149891, 9.98106979885017 50.57656600534072, 9.98150376093987 50.57672072888133)), ((9.984166047894108 50.576997920086804, 9.984811661237101 50.57639594552208, 9.984187544596553 50.5762227524994, 9.984070773132405 50.576163654755455, 9.98395416851367 50.57612740922956, 9.983442218093803 50.57604583686661, 9.982672483942297 50.57589046233731, 9.98280985267337 50.57606652531196, 9.982963953485196 50.576225590441524, 9.98385245565822 50.5770428164268, 9.983935904752407 50.57710116618108, 9.98402275099529 50.576986846114856, 9.984166047894108 50.576997920086804)), ((9.93780878957108 50.57639507833945, 9.937795276294215 50.576393885799405, 9.937569745972077 50.57664143844707, 9.938028944743005 50.576639437551236, 9.93780878957108 50.57639507833945)), ((9.98534148086745 50.576536929470045, 9.984880435404964 50.57641650442203, 9.984243806976702 50.57699862995534, 9.985287722854249 50.57722647296114, 9.985376772528044 50.5772762928254, 9.985923661804359 50.5770931300797, 9.985937985596744 50.576868383747026, 9.985926812316125 50.5767166105999, 9.98534148086745 50.576536929470045)), ((9.985924775894071 50.5771242079693, 9.985912980592193 50.57711519363793, 9.985405846729579 50.577290842157666, 9.98590862555997 50.577577293446524, 9.985924775894071 50.5771242079693)), ((9.924777660234371 50.57891322620466, 9.920423214310944 50.58167996251324, 9.921801470671161 50.58177516430098, 9.922475608779655 50.58185440199788, 9.922570523824257 50.581893574641455, 9.92145328790342 50.58177115987139, 9.920363434671387 50.581713968189526, 9.919489543950213 50.58227782247021, 9.920037742041167 50.58261958193027, 9.921350456056949 50.58286004502588, 9.922399169567864 50.58272765356376, 9.923861429466413 50.58247831786606, 9.924095700130714 50.58242511580182, 9.924316189346753 50.58235201370056, 9.924611596207775 50.58220855575144, 9.925104811971378 50.58201508662482, 9.925840031777941 50.58176982335184, 9.926831084638208 50.58129907407908, 9.92682380672746 50.58128531086672, 9.925872735535846 50.58122673180043, 9.925246304875317 50.58115283560075, 9.924629442670108 50.58105145951064, 9.924037031362879 50.580925770090175, 9.924052147876438 50.58089994130204, 9.9249511180453 50.581079763377566, 9.925569368213349 50.58116692103582, 9.926195408462563 50.581226545319225, 9.926849713063927 50.581256751246855, 9.926537061584334 50.58029246675354, 9.926468362367304 50.57970413708506, 9.926441232157728 50.579031272455616, 9.92649382123377 50.57903041275983, 9.926524260376327 50.579692711071196, 9.927313962141284 50.5797082700635, 9.92731277103274 50.57973522700947, 9.926524752579786 50.57972776569571, 9.926589756064914 50.58029092142982, 9.926914900086631 50.58125942258288, 9.927146018640705 50.5811514509971, 9.928136725554586 50.58049599042841, 9.929095323084654 50.579421843076105, 9.927764987598964 50.57899536539679, 9.926014055488215 50.578130612655805, 9.924777660234371 50.57891322620466)), ((9.967924693588598 50.580260024445266, 9.968046319034057 50.580482560784596, 9.96830981860141 50.58087700802957, 9.968762204662358 50.58145327941729, 9.968931859722902 50.58159804892147, 9.969237557669741 50.58178529177826, 9.969471415070185 50.58188762892594, 9.969769594885605 50.581976465667104, 9.96877721217973 50.581036854423274, 9.96853675442238 50.58058244659175, 9.968505360510498 50.580225528147544, 9.96862214153455 50.57986300301384, 9.968306028037055 50.580104155882466, 9.967924693588598 50.580260024445266)), ((9.9666452675821 50.58130950565109, 9.96627496146969 50.581730267069574, 9.96903298514322 50.5825762237608, 9.96899133622164 50.58237828373779, 9.968684289977775 50.581864815990734, 9.967875750143051 50.580772094930346, 9.967583328233314 50.58047381638354, 9.967231119811775 50.580833022427846, 9.966926739579609 50.58100040015815, 9.9666452675821 50.58130950565109)), ((9.969456074988218 50.58191446761806, 9.969089905907547 50.581744338100094, 9.968887301910685 50.5816116110516, 9.968714101845535 50.581463447664476, 9.968260807582116 50.580886033411865, 9.967872666398657 50.58028586363605, 9.967685523479911 50.580358526919326, 9.96794196185915 50.58075533142808, 9.968759831958799 50.58185471535691, 9.969073769918758 50.582364596138746, 9.969277097347641 50.58257839060102, 9.969557257810498 50.582758095618885, 9.969732425377192 50.582808623181386, 9.970157261737912 50.58267062311186, 9.969880802958528 50.58206733382448, 9.969837439599708 50.582027555291404, 9.969456074988218 50.58191446761806)), ((9.950638260775298 50.582348773474465, 9.950523474524468 50.581326651796594, 9.950541830031062 50.58090028615767, 9.950465749966634 50.58071765679813, 9.95018110823059 50.58096688112783, 9.94969586413109 50.582013737264994, 9.94959403211246 50.58216133345173, 9.949323835462812 50.58228801364905, 9.949389415659928 50.58255842444715, 9.94935623266695 50.582555885110125, 9.949152720676386 50.58234389637358, 9.949112154598957 50.58238291133905, 9.949043804944399 50.582938148170925, 9.948943976468136 50.58336003597525, 9.948873085524149 50.58354035188891, 9.948768674091877 50.58371413486609, 9.948580456200663 50.58393512798008, 9.948491561031318 50.58408363099194, 9.948385775680357 50.584187287100654, 9.948168533535362 50.58431236055484, 9.947818191147364 50.584433485270026, 9.947608634764055 50.58457098156228, 9.947452881577547 50.58478012026292, 9.94733341762682 50.58499766352285, 9.947232241571562 50.58528897463971, 9.947223912861004 50.5854236917042, 9.947252806764554 50.58555594683455, 9.947318456764696 50.58568384611487, 9.947739827322605 50.586360958289596, 9.948356592115179 50.58623110865848, 9.948626367188318 50.58612894251175, 9.948678576680916 50.58591003351626, 9.948760843070747 50.585774566360996, 9.948830580738315 50.58569928176572, 9.948868736339497 50.58561362665194, 9.948860079143309 50.58548195256591, 9.948801606483189 50.58536774532951, 9.94875814433337 50.58522138763381, 9.948757970059132 50.58507261724645, 9.948801088111322 50.58492622100761, 9.948838365917409 50.58485576087782, 9.949234724269084 50.58438713849772, 9.949801452966621 50.583769045639706, 9.949861556731667 50.58368450088503, 9.949949508490246 50.58350999555557, 9.94997801706432 50.58341853947495, 9.950001441668197 50.58323570443312, 9.949989092063372 50.58262018566384, 9.95004179169968 50.582624023008314, 9.95005409694398 50.58323556353889, 9.950030221230225 50.583421758695565, 9.949961956568655 50.58360397182808, 9.949850764400058 50.58377772486751, 9.949282907377858 50.58439713128912, 9.948889027858202 50.58486277797434, 9.948852395778927 50.58493177569256, 9.948810648626202 50.5850735423657, 9.94881082077049 50.58522043628607, 9.948852900359185 50.58536216175523, 9.94891175673953 50.58547699516291, 9.948923110918738 50.58561963238158, 9.949014045651637 50.585657897153766, 9.94914533980459 50.58568596219125, 9.949284682618842 50.585686110192846, 9.94953228910724 50.58564908067345, 9.949885745472745 50.58558162755903, 9.950389576861445 50.585446788394115, 9.95040918724429 50.585471811259104, 9.949906781365756 50.58560613921513, 9.949302234777356 50.58571171528698, 9.949101099618368 50.585707975191895, 9.948911627957694 50.585653188660515, 9.948730319677397 50.5859146036837, 9.948683219368592 50.58610041661092, 9.949323302194756 50.58592240396742, 9.950062447206706 50.58579856625121, 9.950412262294865 50.585832555201684, 9.95098697534051 50.585780772277346, 9.950956545638858 50.58553634870327, 9.950462737326841 50.58479639182127, 9.950412181665262 50.5847495852111, 9.950481284767196 50.58447552655331, 9.950559566064538 50.58429487430342, 9.950630556154557 50.58414882509927, 9.950911938866305 50.583708792382765, 9.951081257591648 50.58323421104324, 9.951089508324824 50.583069473234524, 9.95101824213445 50.58292992884277, 9.950638260775298 50.582348773474465)), ((9.961944723992294 50.58242530193832, 9.962883831265657 50.582554922105565, 9.964096201159146 50.58289600619792, 9.96539812907102 50.58309428389585, 9.965982679613413 50.58337863227307, 9.966177018998904 50.58326343723872, 9.965550439194368 50.58308025522537, 9.961926176436716 50.58213001924701, 9.960764614135918 50.58186344943064, 9.960721670427086 50.581896210407166, 9.961324062533313 50.582143977592416, 9.961944723992294 50.58242530193832)), ((9.919451605627467 50.58229673930815, 9.919030249192783 50.58256972469077, 9.91995255014607 50.58316182573068, 9.920466502098487 50.58302630467916, 9.92121581545339 50.582875179149234, 9.921209166864122 50.58286277555009, 9.92000766154906 50.58264272067997, 9.919451605627467 50.58229673930815)), ((9.92162013868322 50.58286223355442, 9.921615166707157 50.58287786031433, 9.922145933051716 50.58334610393564, 9.92271275412172 50.583803594634844, 9.92555250712588 50.58191566319482, 9.925394168302478 50.581956124980785, 9.924898068320054 50.58213671395978, 9.924350141486105 50.58238271943372, 9.924003539657205 50.5824877630854, 9.922416146170535 50.582762340792286, 9.92162013868322 50.58286223355442)), ((9.921550706313285 50.5828734729344, 9.921229415736232 50.58291461782112, 9.920734941264136 50.5830050016718, 9.920252729314951 50.58311959037346, 9.920010898101847 50.58319649673871, 9.920090937301495 50.58324976450515, 9.920373076995546 50.58322193054649, 9.921044004062319 50.58326392457582, 9.9217571430333 50.58354580057846, 9.922595321971531 50.583773815146714, 9.921550706313285 50.5828734729344)), ((9.970592714627992 50.583799721571765, 9.97052989135398 50.58353835665422, 9.970328547991471 50.583020188963545, 9.970176476048627 50.5826961285174, 9.96978239644331 50.5828176413646, 9.970016820120373 50.58320552410798, 9.97018603811684 50.58366367991846, 9.97026644491557 50.584026798427274, 9.970340853724586 50.584675167704155, 9.97009487409278 50.58477351088356, 9.970071251116755 50.584959277917996, 9.968979814657382 50.585702198928296, 9.968859634059124 50.5858218138498, 9.968705443165478 50.58603480215536, 9.968588327939488 50.58614679561801, 9.966918835717674 50.58685682934416, 9.966436651470199 50.5873595621929, 9.967199281070666 50.58705715085179, 9.967719885147522 50.586883632013574, 9.968775966819813 50.586493910200126, 9.970541488701578 50.58600545697485, 9.970159771600457 50.585339032967305, 9.970358163171824 50.58483290510856, 9.970482940352273 50.584635231069235, 9.970662451456548 50.584096062983676, 9.970592714627992 50.583799721571765)), ((9.953930350173692 50.583579804806504, 9.954279956071007 50.58329166715909, 9.95351328323846 50.583056330824306, 9.952959492958222 50.58299795645885, 9.952084051047631 50.58299182512656, 9.951196247578393 50.58295333606041, 9.951241786785847 50.583249798768094, 9.951051019332768 50.58369463793649, 9.950893584944348 50.583915052772966, 9.950757594054663 50.58427704727632, 9.950721620749091 50.5845820636966, 9.950736059618544 50.584586475026065, 9.951002755579507 50.58437151194011, 9.951331617699347 50.584140183329794, 9.951798466607402 50.58388695480588, 9.952215345603983 50.58362680393937, 9.952459366459186 50.58344164387495, 9.9527090270335 50.58321353923978, 9.952784023719582 50.58318773271667, 9.952870004881055 50.58318818094128, 9.952944281030465 50.58321474832465, 9.952993345096617 50.583275663096614, 9.952937334812017 50.583537178495085, 9.952897868295862 50.58394084130645, 9.952926316643671 50.58409807236052, 9.953041522566467 50.584321130432954, 9.953164594237311 50.58445753956153, 9.954453304793482 50.585238532817606, 9.954458834740818 50.585225650881604, 9.954069726507647 50.58448751998702, 9.95400598042818 50.58431639197281, 9.953968473743894 50.58413982411839, 9.953930350173692 50.583579804806504)), ((9.96549181500753 50.58460211589165, 9.96592647964406 50.584479569805424, 9.965979768054307 50.584494455018266, 9.965508411121842 50.584643978280944, 9.965539150630764 50.584784805325825, 9.96552024604827 50.58501070083896, 9.965527647360231 50.58511294103993, 9.965576225085773 50.58531709152648, 9.96566884155538 50.58551477729873, 9.965930410053241 50.58593847461559, 9.965981394758536 50.586071540027035, 9.966001507967079 50.58624917094852, 9.966356052539398 50.586060056445746, 9.966532597294094 50.58599078482291, 9.96671538074135 50.585891515691195, 9.96700229217404 50.585658101241435, 9.96727189672995 50.5853982688947, 9.967608000935977 50.58501337156019, 9.967570016386192 50.58498906847774, 9.967221815587065 50.584579404395086, 9.966841399692658 50.584187386856364, 9.966639477947414 50.584018487655115, 9.966152294289396 50.5837074083881, 9.965918649374405 50.58341972068542, 9.965936696159797 50.583394600380764, 9.965370859982944 50.583119327990765, 9.964072741484156 50.58292109832745, 9.962861358714198 50.58257987736629, 9.96206940731132 50.58247670050362, 9.962897033967275 50.582949816128775, 9.963792095724049 50.58341918512176, 9.964121685980805 50.58356130469041, 9.964574719734575 50.58371961754222, 9.964762851701481 50.58381264767727, 9.964997484362677 50.58398341104562, 9.965207032722102 50.58419941139916, 9.965359621591753 50.584377098611576, 9.96549181500753 50.58460211589165)), ((9.95600691960225 50.58439471796378, 9.955725298567904 50.5842279186417, 9.95542143877085 50.58395914100631, 9.955150425814203 50.583677312329336, 9.954417971044343 50.583333214215294, 9.954329671433396 50.58330845292309, 9.95398251862307 50.583599831865406, 9.95402047041998 50.58413721484031, 9.954075713743613 50.58436940113351, 9.954148709092353 50.584537729462575, 9.954550229435798 50.5852955920254, 9.954893232720021 50.58547050027721, 9.955853117472248 50.58582998870386, 9.956446509229659 50.586113847854605, 9.957175761503636 50.58641442002754, 9.95763461623209 50.58663893236358, 9.958104104033183 50.58689906766496, 9.958373659629489 50.58702341339666, 9.958564762676062 50.58709065250346, 9.958363601601842 50.58692342454698, 9.958127771810513 50.58666896214178, 9.95744258313323 50.58575370698388, 9.95707589572941 50.58517230773418, 9.956844189158879 50.58489644601369, 9.956787168165931 50.58492584738948, 9.956420645863657 50.584712778328516, 9.956084451359224 50.58444081805003, 9.95600691960225 50.58439471796378)), ((9.947630173285757 50.586331590275115, 9.9476420940215 50.586318868538925, 9.947216437010319 50.58562904868139, 9.947167361661302 50.58549359358193, 9.947165806661156 50.58528576261474, 9.94722213680603 50.585102025008254, 9.947325367141612 50.584877009311, 9.947548540781723 50.584555902913074, 9.947734708533785 50.58442689422702, 9.947886589937037 50.584360087925596, 9.948132447921541 50.58428240560391, 9.948329363631345 50.584168798654, 9.948426601814859 50.584074751750556, 9.94851875306853 50.58392217860556, 9.948708147872802 50.58369948420761, 9.948809618161826 50.58352963098489, 9.94887855666585 50.583353105170076, 9.948977025450878 50.58293681603167, 9.94903591209633 50.58240709711161, 9.949099048067001 50.582306390003026, 9.948680893775684 50.5820066599969, 9.948005383664611 50.58209487679815, 9.948119703280394 50.58266276094062, 9.947975826834957 50.58347161633128, 9.947110396458934 50.58409072622936, 9.946759731742643 50.585090603988846, 9.946584896452093 50.58543675681159, 9.94689398993923 50.58585253545623, 9.947630173285757 50.586331590275115)), ((9.959305866882653 50.58460792793852, 9.960590152323364 50.58568427078108, 9.960990601561571 50.585644666116714, 9.961567364457105 50.585381953351806, 9.962223069676988 50.58524169904104, 9.962122117885212 50.5849874571151, 9.962084796793981 50.58478798144416, 9.962083183287943 50.58458714544777, 9.962115888445586 50.584379914826535, 9.96185953003454 50.584228916955794, 9.9607365811687 50.58338960697636, 9.96067247550703 50.58352418208277, 9.960552841525564 50.58388433765159, 9.960447568530103 50.58441197110957, 9.960395873604142 50.584404841031294, 9.960500582073173 50.58388154050525, 9.96069513064598 50.58333385372611, 9.960036752014426 50.58302530402413, 9.959611989519413 50.582847731429744, 9.958711634511277 50.58360774505402, 9.958225532474625 50.58395318617264, 9.958880127256794 50.58426513998472, 9.959305866882653 50.58460792793852)), ((9.965424826728372 50.58463078018011, 9.965412112378235 50.58457390564177, 9.965358572470734 50.58447942280922, 9.965224857346625 50.584298506829775, 9.964943765372936 50.584004377420044, 9.964799655246539 50.58389008147193, 9.964538187891364 50.58374939479074, 9.963913296958676 50.583521080930126, 9.963460538767999 50.58330636235147, 9.964789376289875 50.58481839026973, 9.965424826728372 50.58463078018011)), ((9.960825392705994 50.58338490244439, 9.96081668425317 50.583400471569014, 9.961899576089884 50.584213110830085, 9.962960993077267 50.58483569811614, 9.96293014740368 50.5848557627391, 9.962164488783996 50.58441731542074, 9.962134008261794 50.58461331507809, 9.962143243907402 50.58483611248301, 9.962164818610189 50.584948058194115, 9.96224043930788 50.58516558543212, 9.96229877862868 50.58527929036013, 9.96291819482161 50.58591507576557, 9.962943086389934 50.58627596869026, 9.961493082272044 50.587245088175145, 9.961998733846753 50.58735651768546, 9.962236682258604 50.58735887779718, 9.962391227302417 50.58733384485665, 9.962741026067796 50.58721292328759, 9.96307352069907 50.58707456568273, 9.963387335037996 50.58691947564358, 9.963493043038866 50.58684236697652, 9.963679492516361 50.58667649727626, 9.963892802609832 50.58640372867088, 9.96408533881851 50.586071341583505, 9.964232336803448 50.585704387403034, 9.964263074569109 50.58554653488211, 9.964250447478664 50.58538818671323, 9.964195097049739 50.58523388744295, 9.964067613798152 50.58504878643011, 9.963873020898646 50.584834428575576, 9.963522985811137 50.584539104873436, 9.962877466168987 50.58417375596349, 9.962454261970175 50.583970840577145, 9.962048432826442 50.58380358027409, 9.961186989538884 50.58352124779838, 9.960825392705994 50.58338490244439)), ((9.965446801289568 50.58466467056662, 9.96482096578697 50.58484807303169, 9.964939452134628 50.58498821692644, 9.965277640801522 50.585681019355206, 9.965228974234757 50.58569148642102, 9.964889238597301 50.584995836954235, 9.963334972087749 50.58323326642826, 9.96213396497399 50.582567440333854, 9.961856674179412 50.582429955152506, 9.961283203395833 50.58217193009217, 9.960681914678512 50.581935495406505, 9.959652392532217 50.58281098980952, 9.960811462333716 50.58333529533271, 9.961222822462604 50.583491288020994, 9.96187003331253 50.58369784876246, 9.9622925962916 50.58385529574246, 9.963058817318393 50.584217971965195, 9.963407574334829 50.58441388657532, 9.96370090523042 50.58461325207073, 9.964034395207012 50.58492507994953, 9.964254574468526 50.58521551269505, 9.964309388736584 50.585354180629636, 9.964330521713501 50.58549632987779, 9.964317565370557 50.58563885102198, 9.964257551845185 50.58583402764023, 9.964150408481428 50.586078859702276, 9.963955102647825 50.586416097430956, 9.96373577592565 50.586695277621665, 9.963437042604859 50.58694292645457, 9.96294944610573 50.5871742251447, 9.962420800385956 50.58736589812646, 9.962082361320448 50.58740262096204, 9.962327527547865 50.58862113527756, 9.962472359985947 50.58862643404757, 9.962679232856363 50.588601463633935, 9.963162057556064 50.588447790681336, 9.963796822357775 50.58819652635839, 9.964123144535947 50.58798448367756, 9.964452239225288 50.58774594871678, 9.964957254442226 50.58719856046995, 9.96546091820663 50.586752400764965, 9.965825681870747 50.58638638385321, 9.965918378376692 50.586315486196604, 9.965926886904096 50.586142181953335, 9.965894811433895 50.58601070256201, 9.96560587101237 50.58552607001119, 9.965493447318154 50.58526994921844, 9.96545860366452 50.58509410883317, 9.965453460443012 50.58500634465164, 9.965475191161547 50.5848618114863, 9.965446801289568 50.58466467056662)), ((9.954162156189089 50.585120786637695, 9.953109402149463 50.584477285277536, 9.95298186654766 50.584336842710236, 9.952891577297553 50.584184957885476, 9.952861296034554 50.58410614181239, 9.95283134235691 50.583945303922945, 9.952870689625783 50.58353518013328, 9.952928033592194 50.583287951996745, 9.95291275737774 50.58325266522389, 9.952863483061785 50.583225211638485, 9.952798734384114 50.58322280828572, 9.952754721728983 50.58323930765586, 9.952517310748275 50.58345879029257, 9.952269464242665 50.58364715932517, 9.951847176377763 50.58391087769623, 9.951379512303367 50.584164768700326, 9.951275153148034 50.5842375801323, 9.951588213041937 50.58450464992189, 9.951870392502395 50.58468894237209, 9.95282785261763 50.58519813041021, 9.953852739581317 50.58579349503775, 9.95424834283075 50.58597286508563, 9.954830059438358 50.58615766260408, 9.954967635880159 50.58621530943541, 9.95500915538705 50.58625494018666, 9.955028943178487 50.58629977681603, 9.95507405136466 50.58681445357269, 9.95509529880431 50.586863137434754, 9.955536891021149 50.587084647119276, 9.955895458378917 50.587330637415164, 9.955967990825371 50.58734893018913, 9.957219128641155 50.58740779068307, 9.957621071835284 50.58747057408648, 9.957569129909853 50.5874139032218, 9.956906200301503 50.58702553724944, 9.95602396449268 50.586281643352294, 9.954162156189089 50.585120786637695)), ((9.970032263886068 50.584844817097455, 9.970017624710804 50.58483678533459, 9.96977620031409 50.58503453300955, 9.969363777673635 50.58500467472976, 9.967947304682356 50.585147366935864, 9.967627955177212 50.58501742656383, 9.96746980800062 50.58521334901055, 9.967224083329173 50.58546636835891, 9.966730828722664 50.58589513205144, 9.966551205042611 50.585995550133376, 9.966067100531548 50.586219983686306, 9.96586927610009 50.58636422199566, 9.965482189727187 50.58675571766225, 9.965045244495427 50.5871442287587, 9.96465238045754 50.58756389245849, 9.964493553075224 50.58776008987434, 9.964304333570071 50.588042663822826, 9.964238525547849 50.58818751079381, 9.964959923782224 50.587836588981475, 9.966332001311365 50.58739825689822, 9.966878620699996 50.5868399540867, 9.968551383573226 50.586127735638875, 9.968652553861693 50.58603092362921, 9.968813185450735 50.58581055475913, 9.968937537066552 50.5856865243196, 9.970016238622431 50.58495362815815, 9.970032263886068 50.584844817097455)), ((9.952560726125006 50.586181284957796, 9.95287774611542 50.58649681581769, 9.953622839273152 50.586824757692035, 9.953454116861428 50.587326552056275, 9.95343184797401 50.58757589521099, 9.953952091574491 50.58740022391522, 9.954562624776393 50.58755169654721, 9.955119961103602 50.58754649552747, 9.955868049506808 50.58771314873265, 9.956355323944619 50.5874056135138, 9.956347028703771 50.5873945598393, 9.955951868018701 50.5873743594457, 9.955876112569277 50.58735690774322, 9.955492377431849 50.58709857447074, 9.955057806843097 50.586884721342756, 9.955021439999824 50.58681951615315, 9.954975740536902 50.58630045194566, 9.954955737406166 50.58625864465509, 9.95491568582807 50.58622449391011, 9.954217461154824 50.585994179241254, 9.953815987218567 50.58581211999417, 9.952789295617869 50.58521599413466, 9.951811834996679 50.584695277907215, 9.951495732875609 50.58448147135473, 9.951238538867084 50.584266947770985, 9.951054580519855 50.58439352703235, 9.950724477533832 50.584662212427226, 9.950768969620238 50.584908300060114, 9.950830369056275 50.585094199861516, 9.951122010481148 50.58556063882714, 9.951114570466277 50.5857943944978, 9.951906251924735 50.58571639964516, 9.952265003388908 50.58588627337882, 9.952560726125006 50.586181284957796)), ((9.958145377884465 50.58474974210309, 9.95755866930573 50.58441082025565, 9.95686305533964 50.58488042355954, 9.957097131009562 50.58514656366164, 9.95726953324913 50.5854021610224, 9.957551934282673 50.58587443429919, 9.957995591702725 50.58647822982818, 9.95813451999254 50.58665388409837, 9.958384205206633 50.586926468656145, 9.958663003196952 50.58712842850642, 9.958988346185391 50.58716325386838, 9.959773405830473 50.58723910703275, 9.96022482234193 50.58725780504035, 9.960677126291131 50.58725296082954, 9.961253939615702 50.58722302290182, 9.961249750893623 50.5872096340399, 9.960442288347075 50.58689280202146, 9.95977942993057 50.58631687413216, 9.958845166104188 50.58524192024652, 9.958575344569635 50.584999262826514, 9.958145377884465 50.58474974210309)), ((9.95911715211573 50.585488539273555, 9.959300991760994 50.58569816956028, 9.960518940505406 50.58568547104464, 9.958839552189053 50.584281740193674, 9.95819177736727 50.583979946214114, 9.957600065763623 50.584382590733576, 9.958620840510292 50.58498544928405, 9.958895310170037 50.585232226052405, 9.95911715211573 50.585488539273555)), ((9.953241936133919 50.58669233840787, 9.952836192427478 50.5865136192213, 9.952222450593553 50.58590276507142, 9.951893756586548 50.585744781795825, 9.951111049888443 50.58583008144822, 9.951116738163966 50.58620789846538, 9.951145390901669 50.58638017875643, 9.95122129988182 50.58655981980797, 9.951396240697253 50.5867251186019, 9.95210144162994 50.58731244323623, 9.952341699494777 50.58759972268738, 9.953338056729299 50.58786376335877, 9.953401518821202 50.58732465057421, 9.953563877511636 50.58683907080369, 9.953241936133919 50.58669233840787)), ((9.954911448833991 50.585521496307834, 9.954903404672365 50.58553750944561, 9.956068141028997 50.58626740787244, 9.956950887280037 50.5870115478762, 9.957616043466858 50.587400967284, 9.957827047155648 50.58759424266865, 9.957939615867716 50.58792289134138, 9.957873301949475 50.587915620471215, 9.957778936200521 50.5876467858672, 9.956663265804924 50.58805675141216, 9.956254818640668 50.58816513490979, 9.955891780352479 50.58823144253193, 9.955665725036143 50.58825014287393, 9.955438906565751 50.58824473474086, 9.95513184843876 50.588197089204726, 9.95420430184287 50.58810175339355, 9.954198854257767 50.58811848457947, 9.955132935227628 50.588415727619264, 9.956114522914415 50.58865168005614, 9.95657308287595 50.58871314958636, 9.957044555724133 50.58871539063772, 9.957356499653187 50.58868352886295, 9.957658974502001 50.58862550235782, 9.957889058437676 50.58855892359639, 9.958086917110172 50.58847719062289, 9.958385629797005 50.58829696146652, 9.958914140031185 50.58784782300686, 9.958901306431818 50.58775349516994, 9.95855371466697 50.58713509356983, 9.958060312130371 50.586925663459404, 9.95758702913288 50.58666355978791, 9.957131200300124 50.58644068235694, 9.95640801427276 50.58614285479781, 9.955812358337054 50.58585820938811, 9.954911448833991 50.585521496307834)), ((9.962400985195412 50.58660580159736, 9.962890353079997 50.58626864794622, 9.962868627626172 50.58592552699889, 9.96224121551106 50.58527566483679, 9.961602093515348 50.58540545527082, 9.961024250980783 50.58567156549949, 9.961091946291592 50.58622669705307, 9.961320421979204 50.58656275496288, 9.961890975141861 50.58665876867331, 9.962400985195412 50.58660580159736)), ((9.960241261081753 50.58666639626005, 9.960520176659902 50.58690721171083, 9.961415101371461 50.58723841551699, 9.962335246074147 50.586649426188096, 9.96232528217647 50.58664006717077, 9.961889498099369 50.58668747399706, 9.961277459973317 50.58658481836403, 9.961042132252782 50.58623233097056, 9.960956350630088 50.58567627441775, 9.960519674713105 50.58571960407974, 9.959335456995003 50.585727322718576, 9.959871313500702 50.58634663033993, 9.960241261081753 50.58666639626005)), ((9.95588633123204 50.58774555981637, 9.955111621770104 50.58757304586425, 9.954553591668772 50.587578289194916, 9.953955810814909 50.58743079897874, 9.953422432669575 50.58761296416091, 9.953391502286625 50.58787600197043, 9.954027393570705 50.58805836192857, 9.95514690123656 50.588173069417685, 9.955397691914316 50.588215572574164, 9.955703150828823 50.5882232405992, 9.956129185672715 50.58816353161085, 9.956535473911806 50.58806485550896, 9.956825741203634 50.587970286057036, 9.957776529679961 50.58760974028843, 9.95773775245347 50.58755988732441, 9.957664697096494 50.587518342069934, 9.957368059015492 50.58745248874591, 9.957151663805405 50.58742866898091, 9.956439344075688 50.587399130005316, 9.95588633123204 50.58774555981637)), ((9.962026577002495 50.58740146763396, 9.961600330012354 50.58730364861678, 9.96137457014561 50.58727056152263, 9.961143353979603 50.58726169116293, 9.960455537628103 50.58729424109125, 9.959996663501052 50.587287359662405, 9.958745716429116 50.58718339293399, 9.95917478771295 50.58748877568685, 9.959318743280763 50.587569792181995, 9.959965235548495 50.58779368427969, 9.960403507392483 50.588031459180854, 9.960891534632598 50.58819499254431, 9.96127788433787 50.588419628145346, 9.961431370152612 50.58847931094546, 9.962123205889936 50.58861120663096, 9.961583792448275 50.588516351282124, 9.961429647705224 50.588480644858116, 9.961264646944162 50.58842028520068, 9.960869122422585 50.588195356024315, 9.960379626134767 50.58803325715597, 9.959953399142087 50.587793529847204, 9.959327861690635 50.58759117186721, 9.95869552111801 50.587169988939976, 9.958628584183156 50.58717110018505, 9.958944209570571 50.587751251513914, 9.958989213857627 50.587899507870276, 9.95904404580056 50.588008552345144, 9.959172565092299 50.58817914273109, 9.959343582200969 50.588331679032876, 9.95958672746104 50.588046138964785, 9.96019171900724 50.58818777257566, 9.961076694413476 50.58849978798296, 9.96208727066095 50.58866644994815, 9.962294805188586 50.58879473553851, 9.96227054688228 50.58863149913239, 9.962150682055439 50.58861489639184, 9.962266845787616 50.588609806107904, 9.962026577002495 50.58740146763396)), ((9.962335448414077 50.588632985042835, 9.962326623311437 50.58864374161718, 9.962367058212378 50.58880592199126, 9.963674850573257 50.58840423324727, 9.964186586295007 50.588214864510775, 9.964463058041437 50.58776355430056, 9.964449581243779 50.587759779929584, 9.963815944006894 50.58819048152453, 9.963697997677555 50.58824502663576, 9.963188947636182 50.58844088048241, 9.962675327577442 50.58861173587638, 9.962335448414077 50.588632985042835)), ((9.958442062079863 50.58829183908458, 9.958303768180622 50.58839081011183, 9.958129213316164 50.588488387055406, 9.957752843565686 50.588629583108165, 9.957276082212978 50.58872050236735, 9.957038278265333 50.588741417602314, 9.956718969941685 50.58874561638281, 9.956247860885208 50.58870160417072, 9.955775423686848 50.58860467952761, 9.955147479892904 50.588447132590105, 9.954199864279316 50.58814941061529, 9.9541879714976 50.5881579742272, 9.954424794731073 50.58832985180314, 9.95442563477229 50.588343876192596, 9.956242897086408 50.58963509969879, 9.957979722163765 50.59005783082861, 9.960003250941185 50.590315471785615, 9.960237594351556 50.59000586735405, 9.960294173477651 50.58962343002907, 9.960409304435093 50.58917196019591, 9.959847964364378 50.588797319135914, 9.959222118239921 50.588490935834336, 9.959321111140241 50.58836090734651, 9.959081167374194 50.588141541920436, 9.95900489274225 50.5880384181229, 9.958923937013967 50.587889058482276, 9.958442062079863 50.58829183908458)))"""

# Convert WKT to GeoJSON then to EE geometry
geom = shapely_wkt.loads(WKT)
geojson = json.loads(json.dumps(geom.__geo_interface__))
aoi = ee.Geometry(geojson)
aoi_fc = ee.FeatureCollection([ee.Feature(aoi)])

display_area = ee.Geometry.Rectangle([
    bounds[0] - 0.015,   # lon min - padding
    bounds[1] - 0.015,   # lat min - padding
    bounds[2] + 0.015,   # lon max + padding
    bounds[3] + 0.015    # lat max + padding
])

aoi_edge = ee.Image().byte().paint(featureCollection=aoi_fc, color=1, width=2)

# Compute center
bounds = geom.bounds  # (minx, miny, maxx, maxy)
center_lon = (bounds[0] + bounds[2]) / 2
center_lat = (bounds[1] + bounds[3]) / 2
print(f'AOI loaded: {len(list(geom.geoms))} polygons')
print(f'Bbox: lon [{bounds[0]:.4f}, {bounds[2]:.4f}], lat [{bounds[1]:.4f}, {bounds[3]:.4f}]')
print(f'Map center: {center_lat:.4f}, {center_lon:.4f}')

AOI loaded: 134 polygons
Bbox: lon [9.9190, 9.9862], lat [50.5484, 50.5903]
Map center: 50.5694, 9.9526


In [22]:
def mask_clouds(img):
    scl = img.select('SCL')
    mask = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
               .And(scl.neq(10)).And(scl.neq(11)))
    return img.updateMask(mask).divide(10000).copyProperties(img, ['system:time_start'])

def add_indices(img):
    ndvi = img.normalizedDifference(['B8','B4']).rename('NDVI')
    ndmi = img.normalizedDifference(['B8','B11']).rename('NDMI')
    nbr  = img.normalizedDifference(['B8','B12']).rename('NBR')
    ndre = img.normalizedDifference(['B8A','B5']).rename('NDRE')
    bsi  = img.expression(
        '((B11+B4)-(B8+B2))/((B11+B4)+(B8+B2))',
        {'B11':img.select('B11'),'B4':img.select('B4'),
         'B8':img.select('B8'),'B2':img.select('B2')}
    ).rename('BSI')
    return img.addBands([ndvi, ndmi, nbr, ndre, bsi])

def make_composite(start, end):
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(display_area)
            .filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_MAX))
            .map(mask_clouds).map(add_indices)
            .median())

print('Functions OK')


Functions OK


In [23]:
before = make_composite(BEFORE_START, BEFORE_END)
after  = make_composite(AFTER_START,  AFTER_END)

d_ndvi = after.select('NDVI').subtract(before.select('NDVI')).rename('dNDVI')
d_ndmi = after.select('NDMI').subtract(before.select('NDMI')).rename('dNDMI')
d_nbr  = after.select('NBR') .subtract(before.select('NBR')) .rename('dNBR')
d_ndre = after.select('NDRE').subtract(before.select('NDRE')).rename('dNDRE')
d_bsi  = after.select('BSI') .subtract(before.select('BSI')) .rename('dBSI')

n_b = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
       .filterBounds(display_area).filterDate(BEFORE_START, BEFORE_END)
       .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_MAX)).size().getInfo())
n_a = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
       .filterBounds(display_area).filterDate(AFTER_START, AFTER_END)
       .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_MAX)).size().getInfo())
print(f'BEFORE images: {n_b} | AFTER images: {n_a}')


BEFORE images: 27 | AFTER images: 12


In [24]:
VIS = {
    'RGB':     {'bands':['B4','B3','B2'], 'min':0,    'max':0.3,  'gamma':1.4},
    'FIR':     {'bands':['B8','B4','B3'], 'min':0,    'max':0.5,  'gamma':1.4},
    'B11B8B5': {'bands':['B11','B8','B5'],'min':0,    'max':0.5,  'gamma':1.3},
    'NDVI':    {'min':0.1,  'max':0.9,  'palette':['#d73027','#fc8d59','#ffffbf','#91cf60','#1a9850']},
    'NDMI':    {'min':-0.3, 'max':0.6,  'palette':['#d7191c','#fdae61','#ffffbf','#abd9e9','#2c7bb6']},
    'NBR':     {'min':0.0,  'max':0.8,  'palette':['#d73027','#fc8d59','#ffffbf','#91bfdb','#4575b4']},
    'NDRE':    {'min':0.0,  'max':0.6,  'palette':['#d73027','#fc8d59','#ffffbf','#91cf60','#1a9850']},
    'BSI':     {'min':-0.5, 'max':0.5,  'palette':['#1a9850','#ffffbf','#8c510a']},
    'DELTA':   {'min':-0.4, 'max':0.4,  'palette':['#d73027','#fc8d59','#ffffbf','#91cf60','#1a9850']},
}

LEGENDS = {
    'RGB':     {'title':'🛰️ True Color','info':'B4·B3·B2 — S2-L2A median','gradient':None,'items':[]},
    'FIR':     {'title':'🌡️ False Color IR','info':'B8·B4·B3 — Healthy veg = bright red','gradient':None,'items':[]},
    'B11B8B5': {'title':'🟤 B11/B8/B5 Composite','info':'SWIR·NIR·RedEdge\n🌲 Dark green = intact canopy\n🟣 Magenta = harvested/stressed\n🔵 Blue-grey = bare/wet soil','gradient':None,'items':[]},
    'NDVI': {
        'title':'🌿 NDVI — Canopy vigor / Coupe rase','info':'(B8−B4)/(B8+B4)\n🪓 Coupe rase: NDVI drops to <0.2\n🌱 Éclaircie: patchy drop −0.05–−0.15',
        'gradient':['#d73027','#fc8d59','#ffffbf','#91cf60','#1a9850'],'grad_labels':('0.1 Bare/cut','0.9 Dense forest'),
        'items':[('#1a9850','>0.6 Dense healthy canopy'),('#91cf60','0.4–0.6 Moderate'),('#ffffbf','0.2–0.4 Open/young stand'),('#d73027','<0.2 Bare soil / clearcut')],
        'delta_title':'Δ NDVI','delta_items':[('#1a9850','>+0.10 Recovery'),('#ffffbf','Stable'),('#fc8d59','−0.10–−0.20 Éclaircie?'),('#d73027','<−0.20 Coupe rase?')]},
    'NDMI': {
        'title':'💧 NDMI — Moisture / Dépressage','info':'(B8−B11)/(B8+B11)\n🌿 Dépressage: subtle NDMI drop in young stands\n💧 Drought: NDMI drops before NDVI',
        'gradient':['#d7191c','#fdae61','#ffffbf','#abd9e9','#2c7bb6'],'grad_labels':('−0.3 Dry','0.6 Wet'),
        'items':[('#2c7bb6','>0.3 Well hydrated'),('#abd9e9','0.1–0.3 Good'),('#ffffbf','Neutral'),('#fdae61','Stress'),('#d7191c','<−0.2 Drought/dieback')],
        'delta_title':'Δ NDMI','delta_items':[('#2c7bb6','>+0.08 Gain'),('#ffffbf','Stable'),('#d7191c','<−0.08 Loss ⚠️')]},
    'NBR': {
        'title':'🔥 NBR — Coupe rase / Éclaircie','info':'(B8−B12)/(B8+B12)\n🪓 Coupe rase: drops from >0.5 to <0.1\n🌲 Éclaircie: moderate drop 0.1–0.2',
        'gradient':['#d73027','#fc8d59','#ffffbf','#91bfdb','#4575b4'],'grad_labels':('0.0 Disturbed','0.8 Intact'),
        'items':[('#4575b4','>0.5 Intact forest'),('#91bfdb','0.3–0.5 Lightly modified'),('#ffffbf','0.1–0.3 Moderate disturbance'),('#d73027','<0.1 Clearcut/severe')],
        'delta_title':'Δ NBR','delta_items':[('#4575b4','>+0.10 Regeneration'),('#ffffbf','Stable'),('#d73027','<−0.10 Disturbance ⚠️')]},
    'NDRE': {
        'title':'🟡 NDRE — Dépressage / Early stress','info':'(B8A−B5)/(B8A+B5)\n🌱 Most sensitive to dépressage in young stands\nDetects stress 2–4 weeks before NDVI reacts',
        'gradient':['#d73027','#fc8d59','#ffffbf','#91cf60','#1a9850'],'grad_labels':('0.0 Stressed','0.6 Healthy'),
        'items':[('#1a9850','>0.4 Very healthy'),('#91cf60','0.2–0.4 Healthy'),('#ffffbf','0.1–0.2 Mild stress'),('#d73027','<0.1 Severe stress')],
        'delta_title':'Δ NDRE','delta_items':[('#1a9850','>+0.05 Improvement'),('#ffffbf','Stable'),('#d73027','<−0.05 Decline ⚠️')]},
    'BSI': {
        'title':'🟫 BSI — Bare Soil / Coupe rase confirm','info':'((B11+B4)−(B8+B2))/((B11+B4)+(B8+B2))\n🪓 ΔBSI > +0.2 confirms soil exposure (coupe rase)\nCombine with ΔNBR < −0.2 for high confidence',
        'gradient':['#1a9850','#ffffbf','#8c510a'],'grad_labels':('−0.5 Dense veg','+0.5 Bare soil'),
        'items':[('#1a9850','<−0.1 Dense vegetation'),('#ffffbf','Mixed'),('#d9b27c','0.1–0.3 Partial harvest'),('#8c510a','>0.3 Bare soil / clearcut')],
        'delta_title':'Δ BSI','delta_items':[('#8c510a','>+0.10 Soil exposure ⚠️'),('#ffffbf','Stable'),('#1a9850','<−0.10 Revegetation')]},
}
print('Vis params OK')


Vis params OK


In [25]:
import ipywidgets as widgets
from IPython.display import display

# ── Map — centered on computed AOI center ────────────────────────────────────
Map = geemap.Map(center=[center_lat, center_lon], zoom=13)
Map.add_basemap('SATELLITE')
Map.layout.height = '640px'

# ── Layer registry — order matters: indices FIRST, AOI outline LAST ──────────
LAYER_DEF = [
    ('rgb_b',  before,               'RGB',     'RGB — BEFORE 2025',     True),
    ('rgb_a',  after,                'RGB',     'RGB — AFTER 2026',      False),
    ('fir_b',  before,               'FIR',     'FIR — BEFORE 2025',     False),
    ('fir_a',  after,                'FIR',     'FIR — AFTER 2026',      False),
    ('b11_b',  before,               'B11B8B5', 'B11B8B5 — BEFORE 2025', False),
    ('b11_a',  after,                'B11B8B5', 'B11B8B5 — AFTER 2026',  False),
    ('ndvi_b', before.select('NDVI'),'NDVI',    'NDVI — BEFORE 2025',    False),
    ('ndvi_a', after.select('NDVI'), 'NDVI',    'NDVI — AFTER 2026',     False),
    ('ndvi_d', d_ndvi,               'DELTA',   'NDVI — DELTA',          False),
    ('ndmi_b', before.select('NDMI'),'NDMI',    'NDMI — BEFORE 2025',    False),
    ('ndmi_a', after.select('NDMI'), 'NDMI',    'NDMI — AFTER 2026',     False),
    ('ndmi_d', d_ndmi,               'DELTA',   'NDMI — DELTA',          False),
    ('nbr_b',  before.select('NBR'), 'NBR',     'NBR  — BEFORE 2025',    False),
    ('nbr_a',  after.select('NBR'),  'NBR',     'NBR  — AFTER 2026',     False),
    ('nbr_d',  d_nbr,                'DELTA',   'NBR  — DELTA',          False),
    ('ndre_b', before.select('NDRE'),'NDRE',    'NDRE — BEFORE 2025',    False),
    ('ndre_a', after.select('NDRE'), 'NDRE',    'NDRE — AFTER 2026',     False),
    ('ndre_d', d_ndre,               'DELTA',   'NDRE — DELTA',          False),
    ('bsi_b',  before.select('BSI'), 'BSI',     'BSI  — BEFORE 2025',    False),
    ('bsi_a',  after.select('BSI'),  'BSI',     'BSI  — AFTER 2026',     False),
    ('bsi_d',  d_bsi,                'DELTA',   'BSI  — DELTA',          False),
]

# Add all data layers first
lyr_labels = {}
for key, img, vis_key, label, shown in LAYER_DEF:
    Map.addLayer(img, VIS[vis_key], label, shown)
    lyr_labels[key] = label

# ── AOI outline LAST = always rendered on top ───────────────────────────────
Map.addLayer(aoi_edge, {'palette':['#00ff88'], 'opacity':0.95}, 'AOI outline', True)
chk_aoi = widgets.Checkbox(value=True, description='🟢 AOI outline',
                            style=cb_s, layout=cb_l)

# ── Checkboxes ───────────────────────────────────────────────────────────────
cb_s = {'description_width':'0px'}
cb_l = widgets.Layout(width='215px', margin='1px 0')
defaults = {k: sh for k,_,_,_,sh in LAYER_DEF}
checks = {k: widgets.Checkbox(value=defaults[k], description=lbl, style=cb_s, layout=cb_l)
          for k,_,_,lbl,_ in LAYER_DEF}

def toggle(change):
    for k, lbl in lyr_labels.items():
        lyr = Map.find_layer(lbl)
        if lyr: lyr.visible = checks[k].value
    # AOI togglable — but re-stacks on top when visible
    aoi_lyr = Map.find_layer('AOI outline')
    if aoi_lyr: aoi_lyr.visible = chk_aoi.value

for c in checks.values(): c.observe(toggle, 'value')
chk_aoi.observe(toggle, 'value')

# ── Legend builder ───────────────────────────────────────────────────────────
def legend_html(mode, show_delta):
    if mode not in LEGENDS: return ''
    L = LEGENDS[mode]
    h = f'<div style="font-size:12px;line-height:1.6">'
    h += f'<b style="color:#1a5276">{L["title"]}</b><br>'
    if L.get('info'):
        h += f'<span style="color:#888;font-size:10px;white-space:pre">{L["info"]}</span><br>'
    if L.get('gradient'):
        stops = ','.join(L['gradient'])
        h += f'<div style="background:linear-gradient(to right,{stops});height:10px;border-radius:3px;margin:5px 0 2px"></div>'
        gl = L['grad_labels']
        h += f'<div style="display:flex;justify-content:space-between;font-size:9px;color:#999;margin-bottom:5px"><span>{gl[0]}</span><span>{gl[1]}</span></div>'
    for hx, tx in L.get('items', []):
        h += (f'<div style="display:flex;align-items:center;margin:1px 0">'
              f'<div style="width:14px;height:14px;background:{hx};border-radius:2px;margin-right:5px;flex-shrink:0"></div>'
              f'<span style="font-size:10px">{tx}</span></div>')
    if show_delta and L.get('delta_items'):
        h += f'<div style="font-weight:bold;color:#7d3c98;margin-top:6px;font-size:11px">{L["delta_title"]}</div>'
        for hx, tx in L['delta_items']:
            h += (f'<div style="display:flex;align-items:center;margin:1px 0">'
                  f'<div style="width:14px;height:14px;background:{hx};border-radius:2px;margin-right:5px;flex-shrink:0"></div>'
                  f'<span style="font-size:10px">{tx}</span></div>')
    h += '<div style="margin-top:5px;padding-top:4px;border-top:1px solid #eee">'
    h += '<div style="display:flex;align-items:center">'
    h += '<div style="width:14px;height:3px;background:#00ff88;margin-right:5px"></div>'
    h += '<span style="font-size:10px">AOI outline (always on top)</span></div></div></div>'
    return h

leg_out = widgets.HTML(value=legend_html('RGB', False))
leg_sel = widgets.Dropdown(
    options=[('🛰️ RGB','RGB'),('🌡️ False IR','FIR'),('🟤 B11/B8/B5','B11B8B5'),
             ('🌿 NDVI','NDVI'),('💧 NDMI','NDMI'),('🔥 NBR','NBR'),
             ('🟡 NDRE','NDRE'),('🟫 BSI','BSI')],
    value='RGB', layout=widgets.Layout(width='220px'))
chk_dleg = widgets.Checkbox(value=False, description='Show delta legend',
                             style=cb_s, layout=cb_l)
def upd_leg(c): leg_out.value = legend_html(leg_sel.value, chk_dleg.value)
leg_sel.observe(upd_leg, 'value')
chk_dleg.observe(upd_leg, 'value')

def sec(t, color='#1a5276'):
    return widgets.HTML(
        f'<div style="font-weight:bold;font-size:11px;color:{color};'
        f'border-bottom:1px solid #eee;padding:4px 0 2px;margin-top:6px">{t}</div>')

panel = widgets.VBox([
    widgets.HTML('<div style="font-weight:bold;font-size:14px;color:#1a5276">🌲 France Valley</div>'),
    widgets.HTML('<div style="font-size:10px;color:#999">Germany · Full AOI · 2025 vs 2026</div>'),
    sec('🟢 AOI OUTLINE'),
    chk_aoi,
    sec('🛰️ TRUE COLOR'),
    checks['rgb_b'], checks['rgb_a'],
    sec('🌡️ FALSE COLOR IR'),
    checks['fir_b'], checks['fir_a'],
    sec('🟤 B11/B8/B5 — Canopy structure'),
    checks['b11_b'], checks['b11_a'],
    sec('🌿 NDVI — Canopy / Coupe rase'),
    checks['ndvi_b'], checks['ndvi_a'], checks['ndvi_d'],
    sec('💧 NDMI — Moisture / Dépressage'),
    checks['ndmi_b'], checks['ndmi_a'], checks['ndmi_d'],
    sec('🔥 NBR — Coupe rase / Éclaircie'),
    checks['nbr_b'], checks['nbr_a'], checks['nbr_d'],
    sec('🟡 NDRE — Dépressage / Early stress'),
    checks['ndre_b'], checks['ndre_a'], checks['ndre_d'],
    sec('🟫 BSI — Soil exposure / Coupe rase'),
    checks['bsi_b'], checks['bsi_a'], checks['bsi_d'],
    widgets.HTML('<hr style="border-color:#eee;margin:8px 0">'),
    widgets.HTML('<b style="font-size:11px;color:#1a5276">📖 Legend</b>'),
    leg_sel, chk_dleg, leg_out,
], layout=widgets.Layout(
    width='255px', height='640px', overflow_y='auto',
    padding='10px', border='1px solid #ddd', background_color='#fff'))

toggle(None)
display(widgets.HBox([panel, Map]))


## Detection guide
| Perturbation | Indices | Signal |
|---|---|---|
| **Coupe rase** | ΔNBR < −0.2 + ΔNDVI < −0.3 + ΔBSI > +0.2 | All indices collapse, BSI spikes; B11/B8/B5 turns magenta |
| **Éclaircie** | ΔNDVI −0.05–−0.15 + ΔNBR −0.05–−0.15 | Patchy, no BSI spike; partial gaps in B11/B8/B5 |
| **Dépressage** | ΔNDRE < −0.05 + ΔNDMI < −0.05 | Subtle early signal; young stands only; NDVI barely changes |
| **Canopée état** | NDVI absolute + B11/B8/B5 | NDVI > 0.6 = dense healthy; B11/B8/B5 uniform dark green = intact |

> **Workflow:** RGB → B11/B8/B5 (structural view) → NDVI delta → NBR delta → BSI delta (coupe rase confirmation)